# 22. Personalization Mechanism Diagnostics — Facial Skincare

This notebook combines canonical case-level pipeline contrasts with ranking-outcome-independent history, alignment, item-evidence, headroom, and Brand features. It also incorporates the independent Stage 1 prior-policy control and the controlled LightGBM prior-policy ablation. Main pipeline contrasts use unconditional NDCG@5 at candidate depth 1,000; the stored Stage 1 control enters at its diagnostic depth of 700.

The analyses cover continuous paired deltas, alignment strata, history quantity × coherence, Stage-1 QCHS status, Stage-2 prior-policy status, baseline headroom, Brand heterogeneity, candidate-pool overlap, and an error taxonomy. LightGBM and the Transformer remain separate throughout.

All estimates are descriptive mechanism evidence with bootstrap intervals. The notebook makes no causal attribution, confirmatory decision, or model-selection claim.

## Contrast and Joining Contract

The reader-facing pipeline contrasts are kept separate:

- S1-P − S1-Q: Stage 1 personalization on the ranking endpoint.
- RankP − Base: prior-feature contribution on the fixed query-only pool.
- Full − RankP: pool-policy contrast given prior-aware reranking.
- Full − Base: combined joint-policy difference.

The internal export labels `P1-only_minus_P0`, `P2-P_minus_P2-Q`, `Full_minus_P2-P`, and `Full_minus_P2-Q` correspond to these four contrasts. Full − RankP changes candidate membership, upstream order, and own-pool fitting together; it must not be interpreted as an isolated causal effect of candidate-source variation.

The canonical case-to-query-to-user crosswalk controls every join. Adaptive quartiles are used only when all four cells satisfy the minimum size; otherwise the analysis uses a documented median split or marks the result unavailable.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ==== Imports ====
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [39]:
# ==== Fixed Category and Analysis Contract ====
NOTEBOOK_NAME = "25_personalization_mechanism_diagnostics_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_KEY = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
EXPECTED_QUERY_ONLY_WINNER = "Graph-Hybrid"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
ANALYSIS_DIR = PROJECT_ROOT / "outputs" / "analysis"
BATCH_A_DIR = ANALYSIS_DIR / "stage_allocation_summary"
BATCH_D1_DIR = ANALYSIS_DIR / "stage1_prior_policy_control"
BATCH_D2_DIR = ANALYSIS_DIR / "prior_policy_ablation_lightgbm"
NOTEBOOK14_MANIFEST_PATH = PROJECT_ROOT / "outputs" / "pipeline_aggregate" / "pipeline_manifest.json"
NOTEBOOK16_MANIFEST_PATH = ANALYSIS_DIR / "paired_significance_summary" / "run_manifest.json"
PREFERENCE_ALIGNMENT_MANIFEST_PATH = ANALYSIS_DIR / "preference_alignment" / "alignment_definition_manifest.json"
PREFERENCE_ALIGNMENT_FEATURES_PATH = ANALYSIS_DIR / "preference_alignment" / "case_alignment_features.parquet"
OUT_DIR = ANALYSIS_DIR / "personalization_mechanism_diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Depth contract: headline pool depth = 1000.
#
#   PRIMARY_POOL_DEPTH      Batch D1 (Notebook 21), Batch D2 (Notebook 22),
#                           and Notebook 11/13 out-of-fold artefacts are read
#                           at the amended headline depth 1000.
#   BATCH_A_POOL_DEPTH      Batch A (Notebook 15) also reports the stage
#                           allocation at the amended headline depth 1000.
#
# Every published row is stamped with the depth of the frame that produced it.
PRIMARY_POOL_DEPTH = 1000
BATCH_A_POOL_DEPTH = 1000
PRIMARY_METRIC = "NDCG@5"
PRIMARY_K = 5
PRIMARY_RERANKER = "lightgbm"
SECONDARY_RERANKER = "transformer"
BOOTSTRAP_REPS = 2000
BOOTSTRAP_CONFIDENCE = 0.95
MIN_STRATUM_N = 20
MIN_INTERACTION_CELL_N = 15
FLOAT_TOLERANCE = 1e-12

REQUIRED_STAGE_CONTRASTS = [
    "P1-only_minus_P0",
    "P2-P_minus_P2-Q",
    "Full_minus_P2-P",
    "Full_minus_P2-Q",
]
STAGE_EFFECT_LABELS = {
    "P1-only_minus_P0": "Stage 1 personalization effect",
    "P2-P_minus_P2-Q": "Stage 2 prior-feature effect",
    "Full_minus_P2-P": "Personalized candidate-source effect",
    "Full_minus_P2-Q": "Combined personalization effect",
}
CANONICAL_CONDITION_PAIRS = {
    "P1-only_minus_P0": ("P0", "P1-only"),
    "P2-P_minus_P2-Q": ("P2-Q", "P2-P"),
    "Full_minus_P2-P": ("P2-P", "Full"),
    "Full_minus_P2-Q": ("P2-Q", "Full"),
}

INPUT_PATHS = {
    "batch_a_stage_delta": BATCH_A_DIR / "stage_delta_per_case.parquet",
    "batch_a_paired_summary": BATCH_A_DIR / "stage_delta_paired_summary.csv",
    "batch_a_fallback_qc": BATCH_A_DIR / "cold_fallback_identity_qc.csv",
    "batch_a_metric_authority_qc": BATCH_A_DIR / "aggregate_metric_authority_qc.csv",
    "batch_a_manifest": BATCH_A_DIR / "stage_allocation_manifest.json",
    "notebook14_manifest": NOTEBOOK14_MANIFEST_PATH,
    "notebook16_manifest": NOTEBOOK16_MANIFEST_PATH,
    "preference_alignment_manifest": PREFERENCE_ALIGNMENT_MANIFEST_PATH,
    "preference_alignment_features": PREFERENCE_ALIGNMENT_FEATURES_PATH,
    "batch_d1_per_case": BATCH_D1_DIR / "stage1_prior_policy_per_case.parquet",
    "batch_d1_contrasts": BATCH_D1_DIR / "stage1_prior_policy_contrasts_per_case.parquet",
    "batch_d1_profiles": BATCH_D1_DIR / "stage1_prior_policy_profile_diagnostics.parquet",
    "batch_d1_fallback_qc": BATCH_D1_DIR / "stage1_prior_policy_fallback_identity_qc.csv",
    "batch_d1_manifest": BATCH_D1_DIR / "stage1_prior_policy_manifest.json",
    "batch_d2_per_case": BATCH_D2_DIR / "prior_policy_per_case_metrics.parquet",
    "batch_d2_deltas": BATCH_D2_DIR / "prior_policy_paired_deltas.parquet",
    "batch_d2_profiles": BATCH_D2_DIR / "prior_policy_profile_diagnostics.parquet",
    "batch_d2_fallback_qc": BATCH_D2_DIR / "prior_policy_cold_fallback_identity_qc.csv",
    "batch_d2_manifest": BATCH_D2_DIR / "prior_policy_ablation_manifest.json",
}


NOTEBOOK24_READER_REQUIRED_COLUMNS = [
    "case_id",
    "regime",
]
NOTEBOOK24_READER_REQUIRED_ROLE_COLUMNS = {
    "query_history_coherence": [
        "query_profile_functional_composite",
        "qchs_alignment_mean",
    ],
    "history_quantity": [
        "strict_prior_interaction_count",
        "strict_prior_unique_item_count",
        "qchs_input_prior_interaction_count",
        "qchs_input_prior_unique_item_count",
    ],
    "baseline_item_evidence": [
        "query_target_functional_composite",
        "query_target_catalog_functional_composite",
        "query_specific_family_count_upstream",
        "query_catalog_functional_facet_count",
    ],
    "brand_coherence_or_affinity": [
        "target_brand_seen_in_prior",
        "target_brand_prior_interaction_share",
        "target_brand_prior_known_brand_share",
        "target_brand_is_dominant_prior_brand",
        "dominant_brand_share",
        "brand_entropy_normalized",
        "unique_prior_brand_count",
    ],
}

BATCH_B_PREFERRED_INPUTS = [
    {
        "source_notebook": "23_structurality_analysis_face.ipynb",
        "source_role": "structurality_case_features",
        "manifest_path": ANALYSIS_DIR / "structurality_analysis" / "run_manifest.json",
        "artifact_path": ANALYSIS_DIR / "structurality_analysis" / "preference_structurality_features_by_case.csv",
        "required_role_columns": {
            "query_history_coherence": ["query_profile_functional_alignment"],
            "history_quantity": ["strict_prior_interaction_count", "unique_prior_item_count"],
            "baseline_item_evidence": [
                "query_target_functional_alignment", "query_specific_family_count",
                "source_signal_count", "catalog_relative_idf", "rare_token_share",
            ],
            "brand_coherence_or_affinity": [
                "dominant_brand_share", "brand_entropy", "unique_prior_brand_count",
            ],
        },
    },
    {
        "source_notebook": "24_preference_alignment_face.ipynb",
        "source_role": "preference_alignment_case_features",
        "manifest_path": ANALYSIS_DIR / "preference_alignment" / "alignment_definition_manifest.json",
        "artifact_path": ANALYSIS_DIR / "preference_alignment" / "case_alignment_features.parquet",
        "required_role_columns": {
            "query_history_coherence": ["query_profile_functional_composite", "qchs_alignment_mean"],
            "history_quantity": [
                "strict_prior_interaction_count", "strict_prior_unique_item_count",
                "qchs_input_prior_interaction_count", "qchs_input_prior_unique_item_count",
            ],
            "baseline_item_evidence": [
                "query_target_functional_composite", "query_target_catalog_functional_composite",
                "query_specific_family_count_upstream", "query_catalog_functional_facet_count",
            ],
            "brand_coherence_or_affinity": [
                "target_brand_seen_in_prior", "target_brand_prior_interaction_share",
                "target_brand_prior_known_brand_share", "target_brand_is_dominant_prior_brand",
                "dominant_brand_share", "brand_entropy_normalized", "unique_prior_brand_count",
            ],
        },
    },
]
BATCH_B_MANIFEST_HINTS = [
    *(spec["manifest_path"] for spec in BATCH_B_PREFERRED_INPUTS),
    ANALYSIS_DIR / "structurality_analysis" / "structurality_analysis_manifest.json",
    ANALYSIS_DIR / "preference_alignment" / "run_manifest.json",
    ANALYSIS_DIR / "preference_alignment" / "preference_alignment_manifest.json",
    ANALYSIS_DIR / "preference_alignment_face_item_query" / "run_manifest.json",
    ANALYSIS_DIR / "preference_alignment_face_item_query" / "preference_alignment_manifest.json",
]

BATCH_A_OOF_LINEAGE_SPECS = [
    {
        "model_family": "lightgbm",
        "condition_name": "P2-Q",
        "source_notebook": "11a_no_prior_rerank_lightgbm_face_REVISED_COMMON.ipynb",
        "manifest_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm_no_prior/feature_interpretation_manifest.json",
        "fold_assignment_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm_no_prior/feature_interpretation_fold_assignments.parquet",
        "expected_model_source_condition": "P2-Q",
    },
    {
        "model_family": "lightgbm",
        "condition_name": "P2-P",
        "source_notebook": "11b_prior_rerank_lightgbm_face_REVISED_COMMON.ipynb",
        "manifest_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm/feature_interpretation_manifest.json",
        "fold_assignment_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm/feature_interpretation_fold_assignments.parquet",
        "expected_model_source_condition": "P2-P",
    },
    {
        "model_family": "lightgbm",
        "condition_name": "Full",
        "source_notebook": "11c_personalized_rerank_lightgbm_face_REVISED_COMMON.ipynb",
        # Directory of record per the Notebook 14 SOURCE_RUNS registry is
        # stage2_personalized_rerank/lightgbm_full; the legacy
        # stage2_personalized_retrieval_rerank/lightgbm spelling is retained
        # as a fallback so that neither layout is a hard lineage-QC failure.
        "artifact_dir_candidates": [
            PROJECT_ROOT / "outputs/stage2_personalized_rerank/lightgbm_full",
            PROJECT_ROOT / "outputs/stage2_personalized_retrieval_rerank/lightgbm",
        ],
        "expected_model_source_condition": "P2-P",
    },
    {
        "model_family": "transformer",
        "condition_name": "P2-Q",
        "source_notebook": "13a_no_prior_rerank_transformer_face_REVISED_COMMON.ipynb",
        "manifest_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer_no_prior/feature_interpretation_manifest.json",
        "fold_assignment_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer_no_prior/feature_interpretation_fold_assignments.parquet",
        "expected_model_source_condition": "P2-Q",
    },
    {
        "model_family": "transformer",
        "condition_name": "P2-P",
        "source_notebook": "13b_01_base_rerank_transformer_face.ipynb",
        "manifest_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer/feature_interpretation_manifest.json",
        "fold_assignment_path": PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer/feature_interpretation_fold_assignments.parquet",
        "expected_model_source_condition": "P2-P",
    },
    {
        "model_family": "transformer",
        "condition_name": "Full",
        "source_notebook": "13c_personalized_rerank_transformer_face_REVISED_COMMON.ipynb",
        # Directory of record per the Notebook 14 SOURCE_RUNS registry is
        # stage2_personalized_rerank/transformer_full; the legacy
        # stage2_personalized_retrieval_rerank/transformer spelling is retained
        # as a fallback so that neither layout is a hard lineage-QC failure.
        "artifact_dir_candidates": [
            PROJECT_ROOT / "outputs/stage2_personalized_rerank/transformer_full",
            PROJECT_ROOT / "outputs/stage2_personalized_retrieval_rerank/transformer",
        ],
        "expected_model_source_condition": "P2-P",
    },
]

OUTPUT_PATHS = {
    "case_level": OUT_DIR / "personalization_mechanism_case_level.parquet",
    "continuous_summary": OUT_DIR / "continuous_delta_summary.csv",
    "alignment_strata": OUT_DIR / "alignment_strata_summary.csv",
    "history_coherence": OUT_DIR / "history_quantity_coherence_interaction.csv",
    "stage1_qchs_status": OUT_DIR / "stage1_qchs_active_fallback_summary.csv",
    "prior_policy_heterogeneity": OUT_DIR / "qchs_all_prior_heterogeneity.csv",
    "headroom": OUT_DIR / "baseline_item_evidence_headroom.csv",
    "brand": OUT_DIR / "brand_coherence_heterogeneity.csv",
    "error_taxonomy_case": OUT_DIR / "continuous_delta_error_taxonomy.parquet",
    "error_taxonomy_summary": OUT_DIR / "continuous_delta_error_taxonomy_summary.csv",
    "strong_history_premium": OUT_DIR / "strong_history_premium_summary.csv",
    "feature_contract": OUT_DIR / "mechanism_feature_contract.csv",
    "join_qc": OUT_DIR / "join_grain_qc.csv",
    "lineage_qc": OUT_DIR / "lineage_leakage_qc.csv",
    "stratification_qc": OUT_DIR / "stratification_qc.csv",
    "candidate_overlap_qc": OUT_DIR / "candidate_overlap_resolution_qc.csv",
    "manifest": OUT_DIR / "run_manifest.json",
}

print("Category:", CATEGORY_LABEL)
print(
    "Primary contract:", PRIMARY_METRIC,
    "| Batch A (Notebook 15) pool depth", BATCH_A_POOL_DEPTH,
    "| Batch D1/D2 prior-policy pool depth", PRIMARY_POOL_DEPTH,
)
print("Output directory:", OUT_DIR)


Category: Facial Skincare
Primary contract: NDCG@5 | Batch A (Notebook 15) pool depth 1000 | Batch D1/D2 prior-policy pool depth 1000
Output directory: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/personalization_mechanism_diagnostics


In [40]:
# ==== Validation, loading, and Bootstrap Helpers ====
def require_columns(frame, columns, label):
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique(frame, keys, label):
    require_columns(frame, keys, label)
    duplicated = frame.duplicated(keys, keep=False)
    if duplicated.any():
        sample = frame.loc[duplicated, keys].head(10).to_dict("records")
        raise RuntimeError(f"{label} violates {keys} grain; sample={sample}")


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {path}")


def boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    normalized = series.astype("string").str.strip().str.lower()
    allowed = {"true", "false", "1", "0", "yes", "no", "y", "n", ""}
    unexpected = sorted(set(normalized.dropna()).difference(allowed))
    if unexpected:
        raise RuntimeError(f"Unexpected boolean values: {unexpected[:10]}")
    return normalized.isin({"true", "1", "yes", "y"})


def stable_seed(*parts):
    payload = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**32 - 1)


def ndcg_at_5_from_rank(rank):
    if pd.isna(rank):
        return 0.0
    rank = int(rank)
    if rank < 1 or rank > PRIMARY_K:
        return 0.0
    return float(1.0 / math.log2(rank + 1.0))


def user_cluster_bootstrap_mean(frame, value_col, seed_label):
    work = frame[["user_id", value_col]].copy()
    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
    work = work.dropna(subset=["user_id", value_col])
    if work.empty:
        return np.nan, np.nan
    clusters = work.groupby("user_id", observed=True)[value_col].agg(["sum", "count"])
    sums = clusters["sum"].to_numpy(dtype=float)
    counts = clusters["count"].to_numpy(dtype=float)
    rng = np.random.default_rng(stable_seed(CATEGORY_ID, seed_label))
    draws = np.empty(BOOTSTRAP_REPS, dtype=float)
    for start in range(0, BOOTSTRAP_REPS, 250):
        stop = min(start + 250, BOOTSTRAP_REPS)
        index = rng.integers(0, len(clusters), size=(stop - start, len(clusters)))
        draws[start:stop] = sums[index].sum(axis=1) / counts[index].sum(axis=1)
    alpha = (1.0 - BOOTSTRAP_CONFIDENCE) / 2.0
    return tuple(np.quantile(draws, [alpha, 1.0 - alpha]).astype(float))


def continuous_summary(frame, value_col, seed_label):
    values = pd.to_numeric(frame[value_col], errors="coerce").dropna().to_numpy(dtype=float)
    ci_low, ci_high = user_cluster_bootstrap_mean(frame, value_col, seed_label)
    return {
        "n_cases": int(len(values)),
        "n_users": int(frame.loc[pd.to_numeric(frame[value_col], errors="coerce").notna(), "user_id"].nunique()),
        "mean_delta_ndcg_at_5": float(np.mean(values)) if len(values) else np.nan,
        "median_delta_ndcg_at_5": float(np.median(values)) if len(values) else np.nan,
        "bootstrap_ci_95_low": ci_low,
        "bootstrap_ci_95_high": ci_high,
        "positive_delta_rate_descriptive": float(np.mean(values > 0.0)) if len(values) else np.nan,
        "zero_delta_rate_descriptive": float(np.mean(values == 0.0)) if len(values) else np.nan,
        "negative_delta_rate_descriptive": float(np.mean(values < 0.0)) if len(values) else np.nan,
        "bootstrap_unit": "user_id cluster over paired case-level deltas",
        "bootstrap_repetitions": BOOTSTRAP_REPS,
    }


def cluster_bootstrap_group_gap(frame, value_col, group_col, left_group, right_group, seed_label):
    work = frame[["user_id", value_col, group_col]].copy()
    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
    work = work.dropna(subset=["user_id", value_col, group_col])
    work = work.loc[work[group_col].isin([left_group, right_group])]
    if work.empty or set(work[group_col]) != {left_group, right_group}:
        return np.nan, np.nan, np.nan
    users = pd.Index(sorted(work["user_id"].astype(str).unique()))
    aggregates = work.groupby([work["user_id"].astype(str), group_col], observed=True)[value_col].agg(["sum", "count"])
    sum_wide = aggregates["sum"].unstack(group_col).reindex(users).fillna(0.0)
    count_wide = aggregates["count"].unstack(group_col).reindex(users).fillna(0.0)
    observed = (
        work.loc[work[group_col].eq(left_group), value_col].mean()
        - work.loc[work[group_col].eq(right_group), value_col].mean()
    )
    rng = np.random.default_rng(stable_seed(CATEGORY_ID, seed_label))
    draws = []
    for _ in range(BOOTSTRAP_REPS):
        index = rng.integers(0, len(users), size=len(users))
        left_count = count_wide[left_group].to_numpy(dtype=float)[index].sum()
        right_count = count_wide[right_group].to_numpy(dtype=float)[index].sum()
        if left_count and right_count:
            left_mean = sum_wide[left_group].to_numpy(dtype=float)[index].sum() / left_count
            right_mean = sum_wide[right_group].to_numpy(dtype=float)[index].sum() / right_count
            draws.append(left_mean - right_mean)
    if not draws:
        return float(observed), np.nan, np.nan
    alpha = (1.0 - BOOTSTRAP_CONFIDENCE) / 2.0
    low, high = np.quantile(np.asarray(draws), [alpha, 1.0 - alpha])
    return float(observed), float(low), float(high)


def cluster_bootstrap_four_cell_interaction(frame, value_col, cell_col, seed_label):
    cells = [
        "low_history__low_coherence", "low_history__high_coherence",
        "high_history__low_coherence", "high_history__high_coherence",
    ]
    work = frame[["user_id", value_col, cell_col]].copy()
    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
    work = work.dropna(subset=["user_id", value_col, cell_col])
    if set(work[cell_col]) != set(cells):
        return np.nan, np.nan, np.nan
    users = pd.Index(sorted(work["user_id"].astype(str).unique()))
    aggregates = work.groupby([work["user_id"].astype(str), cell_col], observed=True)[value_col].agg(["sum", "count"])
    sum_wide = aggregates["sum"].unstack(cell_col).reindex(index=users, columns=cells).fillna(0.0)
    count_wide = aggregates["count"].unstack(cell_col).reindex(index=users, columns=cells).fillna(0.0)
    means = work.groupby(cell_col, observed=True)[value_col].mean()
    observed = float(
        means["high_history__high_coherence"]
        - means["high_history__low_coherence"]
        - means["low_history__high_coherence"]
        + means["low_history__low_coherence"]
    )
    rng = np.random.default_rng(stable_seed(CATEGORY_ID, seed_label))
    draws = []
    sums = sum_wide.to_numpy(dtype=float)
    counts = count_wide.to_numpy(dtype=float)
    for _ in range(BOOTSTRAP_REPS):
        index = rng.integers(0, len(users), size=len(users))
        sampled_counts = counts[index].sum(axis=0)
        if np.all(sampled_counts > 0):
            sampled_means = sums[index].sum(axis=0) / sampled_counts
            draws.append(sampled_means[3] - sampled_means[2] - sampled_means[1] + sampled_means[0])
    if not draws:
        return observed, np.nan, np.nan
    alpha = (1.0 - BOOTSTRAP_CONFIDENCE) / 2.0
    low, high = np.quantile(np.asarray(draws), [alpha, 1.0 - alpha])
    return observed, float(low), float(high)


def spearman_descriptive(frame, feature, value_col="delta_ndcg_at_5"):
    pair = frame[[feature, value_col]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(pair) < 3 or pair[feature].nunique() < 2 or pair[value_col].nunique() < 2:
        return np.nan, int(len(pair))
    correlation = pair[feature].rank(method="average").corr(pair[value_col].rank(method="average"))
    return float(correlation), int(len(pair))


def normalized_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def recursive_paths(payload):
    if isinstance(payload, dict):
        for value in payload.values():
            yield from recursive_paths(value)
    elif isinstance(payload, list):
        for value in payload:
            yield from recursive_paths(value)
    elif isinstance(payload, str) and Path(payload).suffix.lower() in {".parquet", ".csv"}:
        yield Path(payload)

class Notebook24DependencyError(RuntimeError):
    """Raised when required Notebook 24 reader-side artifacts fail their contract."""


def sha256_file(path, chunk_bytes=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def recursive_string_values(payload):
    if isinstance(payload, dict):
        for value in payload.values():
            yield from recursive_string_values(value)
    elif isinstance(payload, list):
        for value in payload:
            yield from recursive_string_values(value)
    elif isinstance(payload, str):
        yield payload


def manifest_mentions_artifact(manifest, expected_path):
    expected_name = Path(expected_path).name
    expected_stem = Path(expected_path).stem
    normalized_expected = {normalized_name(expected_name), normalized_name(expected_stem)}
    for value in recursive_string_values(manifest):
        normalized_value = normalized_name(value)
        if any(token and token in normalized_value for token in normalized_expected):
            return True
    return False


def extract_notebook24_feature_sha256(manifest, feature_path):
    expected_name = normalized_name(Path(feature_path).name)
    expected_stem = normalized_name(Path(feature_path).stem)
    candidates = []

    def walk(payload, context_matches=False):
        if isinstance(payload, dict):
            scalar_text = " ".join(
                normalized_name(value)
                for value in payload.values()
                if isinstance(value, str)
            )
            local_context_matches = context_matches or expected_name in scalar_text or expected_stem in scalar_text
            for key, value in payload.items():
                normalized_key = normalized_name(key)
                key_matches_artifact = expected_name in normalized_key or expected_stem in normalized_key
                if isinstance(value, str):
                    looks_like_sha = bool(re.fullmatch(r"[0-9a-fA-F]{64}", value.strip()))
                    key_is_hash = "sha256" in normalized_key or normalized_key.endswith("hash") or "checksum" in normalized_key
                    if looks_like_sha and key_is_hash and (local_context_matches or key_matches_artifact):
                        candidates.append(value.strip().lower())
                walk(value, local_context_matches or key_matches_artifact)
        elif isinstance(payload, list):
            for value in payload:
                walk(value, context_matches)

    walk(manifest)
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) > 1:
        raise Notebook24DependencyError(
            "Notebook 24 alignment manifest declares multiple SHA256 values for "
            f"{Path(feature_path).name}; observed={candidates}"
        )
    return candidates[0] if candidates else None


def select_notebook24_feature_frame(batch_b_frames, expected_path):
    expected = Path(expected_path)
    matches = [
        frame
        for artifact_path, frame, _feature_columns in batch_b_frames
        if Path(artifact_path) == expected
    ]
    if len(matches) != 1:
        raise Notebook24DependencyError(
            "Notebook 24 case_alignment_features.parquet was not loaded as an accepted "
            f"Batch B case-feature artifact; observed_matches={len(matches)}"
        )
    return matches[0].copy()


def validate_notebook24_reader_contract_core(
    manifest,
    feature_frame,
    case_crosswalk,
    category_id,
    declared_feature_sha256,
    observed_feature_sha256,
    required_columns=None,
    required_role_columns=None,
):
    required_columns = list(required_columns or NOTEBOOK24_READER_REQUIRED_COLUMNS)
    required_role_columns = dict(required_role_columns or NOTEBOOK24_READER_REQUIRED_ROLE_COLUMNS)
    if str(manifest.get("category_id")) != str(category_id):
        raise Notebook24DependencyError(
            f"Notebook 24 preference-alignment category mismatch: {manifest.get('category_id')}"
        )
    if not isinstance(feature_frame, pd.DataFrame) or feature_frame.empty:
        raise Notebook24DependencyError("Notebook 24 case_alignment_features.parquet is empty or unreadable.")
    missing_base = [column for column in required_columns if column not in feature_frame.columns]
    if missing_base:
        raise Notebook24DependencyError(f"Notebook 24 alignment features are missing required columns: {missing_base}")
    missing_roles = {
        role: columns
        for role, columns in required_role_columns.items()
        if not any(column in feature_frame.columns for column in columns)
    }
    if missing_roles:
        raise Notebook24DependencyError(
            "Notebook 24 alignment features are missing required role columns: "
            + json.dumps(missing_roles, ensure_ascii=False, sort_keys=True)
        )
    if "category_id" in feature_frame.columns:
        observed_categories = set(feature_frame["category_id"].astype(str).dropna())
        if observed_categories != {str(category_id)}:
            raise Notebook24DependencyError(
                f"Notebook 24 feature-table category mismatch: {sorted(observed_categories)}"
            )
    work = feature_frame.copy()
    work["case_id"] = work["case_id"].astype(str)
    if work.duplicated("case_id").any():
        sample = work.loc[work.duplicated("case_id", keep=False), "case_id"].head(10).tolist()
        raise Notebook24DependencyError(f"Notebook 24 alignment features contain duplicate case_id values: {sample}")
    if work.duplicated().any():
        raise Notebook24DependencyError("Notebook 24 alignment features contain duplicate rows.")
    canonical = case_crosswalk[["case_id", "regime"]].copy()
    canonical["case_id"] = canonical["case_id"].astype(str)
    canonical["regime"] = canonical["regime"].astype(str).str.lower()
    expected_cases = set(canonical["case_id"])
    observed_cases = set(work["case_id"])
    missing_cases = sorted(expected_cases.difference(observed_cases))
    unexpected_cases = sorted(observed_cases.difference(expected_cases))
    if len(observed_cases) != len(expected_cases):
        raise Notebook24DependencyError(
            "Notebook 24 alignment feature benchmark case count mismatch: "
            f"expected={len(expected_cases)}, observed={len(observed_cases)}"
        )
    if missing_cases or unexpected_cases:
        raise Notebook24DependencyError(
            "Notebook 24 alignment feature case universe mismatch: "
            + json.dumps({
                "missing_required_cases": missing_cases[:10],
                "unexpected_additional_cases": unexpected_cases[:10],
                "missing_required_case_count": len(missing_cases),
                "unexpected_additional_case_count": len(unexpected_cases),
            }, ensure_ascii=False, sort_keys=True)
        )
    merged = work[["case_id", "regime"]].merge(
        canonical.rename(columns={"regime": "canonical_regime"}),
        on="case_id",
        how="left",
        validate="one_to_one",
    )
    regime_mismatch = int(
        (~merged["regime"].astype(str).str.lower().eq(merged["canonical_regime"])).sum()
    )
    if regime_mismatch:
        raise Notebook24DependencyError(
            f"Notebook 24 alignment feature regime lineage mismatch count: {regime_mismatch}"
        )
    if not observed_feature_sha256:
        raise Notebook24DependencyError("Notebook 24 alignment feature observed SHA256 evidence is missing.")

    declared_feature_sha256_normalized = (
        str(declared_feature_sha256).lower()
        if declared_feature_sha256
        else None
    )
    observed_feature_sha256_normalized = str(observed_feature_sha256).lower()

    if (
        declared_feature_sha256_normalized is not None
        and declared_feature_sha256_normalized != observed_feature_sha256_normalized
    ):
        raise Notebook24DependencyError(
            "Notebook 24 alignment feature SHA256 mismatch: "
            f"manifest={declared_feature_sha256}, observed={observed_feature_sha256}"
        )
    return {
        "manifest_category_id": str(manifest.get("category_id")),
        "feature_case_count": int(len(observed_cases)),
        "required_columns": required_columns,
        "required_role_columns": required_role_columns,
        "declared_case_alignment_features_sha256": declared_feature_sha256_normalized,
        "observed_case_alignment_features_sha256": observed_feature_sha256_normalized,
        "feature_sha256_manifest_match": (
            declared_feature_sha256_normalized == observed_feature_sha256_normalized
            if declared_feature_sha256_normalized is not None
            else None
        ),
        "feature_sha256_manifest_declared": declared_feature_sha256_normalized is not
        None,
        "case_universe_match": True,
        "regime_lineage_match": True,
        "notebook24_role": "mechanism_feature_input_only",
    }


def validate_notebook24_reader_contract(
    manifest,
    manifest_path,
    feature_frame,
    feature_path,
    case_crosswalk,
    category_id,
):
    manifest_path = Path(manifest_path)
    feature_path = Path(feature_path)
    if not manifest_path.exists():
        raise Notebook24DependencyError(f"Missing Notebook 24 alignment manifest: {manifest_path}")
    if not feature_path.exists():
        raise Notebook24DependencyError(f"Missing Notebook 24 alignment feature parquet: {feature_path}")
    if not manifest_mentions_artifact(manifest, feature_path):
        raise Notebook24DependencyError(
            f"Notebook 24 manifest does not declare {feature_path.name} as an output artifact."
        )
    declared_feature_sha256 = extract_notebook24_feature_sha256(manifest, feature_path)
    observed_feature_sha256 = sha256_file(feature_path)
    contract = validate_notebook24_reader_contract_core(
        manifest=manifest,
        feature_frame=feature_frame,
        case_crosswalk=case_crosswalk,
        category_id=category_id,
        declared_feature_sha256=declared_feature_sha256,
        observed_feature_sha256=observed_feature_sha256,
    )
    contract["manifest_path"] = str(manifest_path)
    contract["manifest_sha256"] = sha256_file(manifest_path)
    contract["feature_path"] = str(feature_path)
    return contract


## Data and lineage

In [41]:
# ==== Load Standardized Batch A, D1, and D2 Outputs ====
missing_notebook24_outputs = []
if not Path(INPUT_PATHS["preference_alignment_manifest"]).exists():
    missing_notebook24_outputs.append(str(INPUT_PATHS["preference_alignment_manifest"]))
if not Path(INPUT_PATHS["preference_alignment_features"]).exists():
    missing_notebook24_outputs.append(str(INPUT_PATHS["preference_alignment_features"]))
if missing_notebook24_outputs:
    raise Notebook24DependencyError(
        "Notebook 25 requires Notebook 24 reader-side outputs: "
        "alignment_definition_manifest.json and case_alignment_features.parquet. Missing: "
        + json.dumps(missing_notebook24_outputs, indent=2)
    )

missing_required = [str(path) for path in INPUT_PATHS.values() if not Path(path).exists()]
if missing_required:
    raise FileNotFoundError(
        "Required standardized upstream outputs are missing. Execute Batch A, then Batch B, "
        "then Batch D1, then Batch D2 for this category before Batch E. Missing: "
        + json.dumps(missing_required, indent=2)
    )

batch_a_delta = read_table(INPUT_PATHS["batch_a_stage_delta"])
batch_a_summary = read_table(INPUT_PATHS["batch_a_paired_summary"])
batch_a_fallback_qc = read_table(INPUT_PATHS["batch_a_fallback_qc"])
batch_a_authority_qc = read_table(INPUT_PATHS["batch_a_metric_authority_qc"])
batch_a_manifest = load_json(INPUT_PATHS["batch_a_manifest"])
notebook14_manifest = load_json(INPUT_PATHS["notebook14_manifest"])
notebook16_manifest = load_json(INPUT_PATHS["notebook16_manifest"])
preference_alignment_manifest = load_json(INPUT_PATHS["preference_alignment_manifest"])
if notebook14_manifest.get("run_status") != "SUCCESS" or notebook14_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 14 canonical outputs are not ready for downstream analysis.")
if str(notebook14_manifest.get("category_id")) != CATEGORY_ID:
    raise RuntimeError("Notebook 14 category mismatch.")
if str(notebook16_manifest.get("category_id", CATEGORY_ID)) != CATEGORY_ID:
    raise RuntimeError("Notebook 16 category mismatch.")
if preference_alignment_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 24 preference-alignment category mismatch.")

notebook14_canonical_raw_sha256 = (
    notebook14_manifest.get("canonical_raw_sha256")
    or notebook14_manifest.get("output_sha256", {}).get("canonical_raw")
    or notebook14_manifest.get("output_hashes", {}).get("canonical_raw")
)
if not notebook14_canonical_raw_sha256:
    raise RuntimeError("Notebook 14 manifest lacks canonical raw metric SHA256; refusing stale lineage.")
batch_a_notebook14_sha256 = (
    batch_a_manifest.get("notebook14_canonical_raw_sha256")
    or batch_a_manifest.get("input_sha256", {}).get("canonical_per_case_metrics")
    or batch_a_manifest.get("input_sha256", {}).get("notebook14_canonical_raw")
    or batch_a_manifest.get("input_sha256", {}).get("pipeline_canonical_per_case_metrics")
)
if batch_a_notebook14_sha256 != notebook14_canonical_raw_sha256:
    raise RuntimeError("Batch A/Notebook 15 stage effects are stale relative to Notebook 14 canonical metrics.")
notebook16_input_sha256 = notebook16_manifest.get("input_sha256", {})
notebook16_notebook14_sha256 = (
    notebook16_input_sha256.get("canonical_per_case_metrics")
    or notebook16_input_sha256.get("notebook14_canonical_raw")
    or notebook16_input_sha256.get("pipeline_canonical_per_case_metrics")
)
if notebook16_notebook14_sha256 != notebook14_canonical_raw_sha256:
    raise RuntimeError("Notebook 16 inferential inputs are stale relative to Notebook 14 canonical metrics.")

batch_d1_per_case = read_table(INPUT_PATHS["batch_d1_per_case"])
batch_d1_delta = read_table(INPUT_PATHS["batch_d1_contrasts"])
batch_d1_profiles = read_table(INPUT_PATHS["batch_d1_profiles"])
batch_d1_fallback_qc = read_table(INPUT_PATHS["batch_d1_fallback_qc"])
batch_d1_manifest = load_json(INPUT_PATHS["batch_d1_manifest"])

batch_d2_per_case = read_table(INPUT_PATHS["batch_d2_per_case"])
batch_d2_delta = read_table(INPUT_PATHS["batch_d2_deltas"])
batch_d2_profiles = read_table(INPUT_PATHS["batch_d2_profiles"])
batch_d2_fallback_qc = read_table(INPUT_PATHS["batch_d2_fallback_qc"])
batch_d2_manifest = load_json(INPUT_PATHS["batch_d2_manifest"])

def resolve_available_pool_depth(frame, label, preferred_depth):
    depths = sorted(
        pd.to_numeric(frame["pool_depth"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if int(preferred_depth) in depths:
        return int(preferred_depth)
    if len(depths) == 1:
        print(
            f"WARNING: {label} has no rows at pool depth {preferred_depth}; "
            f"using available source depth {depths[0]}."
        )
        return int(depths[0])
    raise RuntimeError(
        f"{label} has no usable row at preferred pool depth {preferred_depth}; "
        f"available_depths={depths}"
    )


BATCH_D1_POOL_DEPTH = resolve_available_pool_depth(
    batch_d1_per_case, "Batch D1 per-case metrics", PRIMARY_POOL_DEPTH
)
BATCH_D2_POOL_DEPTH = resolve_available_pool_depth(
    batch_d2_per_case, "Batch D2 per-case metrics", PRIMARY_POOL_DEPTH
)

require_columns(batch_d1_profiles, ["case_id", "qchs_active"], "Notebook 21 profile diagnostics")
require_columns(batch_d2_profiles, ["case_id", "qchs_active"], "Notebook 22 profile diagnostics")
batch_d1_profiles = batch_d1_profiles.copy()
batch_d2_profiles = batch_d2_profiles.copy()
batch_d1_profiles["stage1_qchs_active"] = boolean_series(batch_d1_profiles["qchs_active"])
batch_d2_profiles["stage2_qchs_filtered_prior_active"] = boolean_series(batch_d2_profiles["qchs_active"])
required_batch_d2_contract = "controlled_lightgbm_prior_policy_ablation_v2_primary_novel_item"
if batch_d2_manifest.get("contract_version") != required_batch_d2_contract:
    raise RuntimeError(
        "Batch D2 prior-policy contract is stale or incompatible: "
        f"{batch_d2_manifest.get('contract_version')}"
    )
if batch_d2_manifest.get("primary_prior_policy_exact_item_block_excluded") is not True:
    raise RuntimeError("Batch D2 primary prior-policy models include or do not prove exclusion of the exact-item block.")
if batch_d2_manifest.get("profile_family_support_feature") != "candidate_profile_family_support":
    raise RuntimeError("Batch D2 still uses ambiguous seen-item semantics for broad profile support.")
if batch_d2_manifest.get("profile_family_support_semantic_version") != "candidate_profile_family_support_v1":
    raise RuntimeError("Batch D2 profile-family-support semantic version mismatch.")

require_columns(
    batch_d1_per_case,
    ["category_id", "case_id", "query_id", "user_id", "regime", "prior_policy", "pool_depth", "target_rank", "ndcg_at_5"],
    "Batch D1 per-case metrics",
)
batch_d1_per_case = batch_d1_per_case.loc[
    batch_d1_per_case["category_id"].astype(str).eq(CATEGORY_ID)
    & pd.to_numeric(batch_d1_per_case["pool_depth"], errors="coerce").eq(BATCH_D1_POOL_DEPTH)
].copy()
require_unique(batch_d1_per_case, ["case_id", "prior_policy"], "Batch D1 per-case metrics")
d1_reconstructed = batch_d1_per_case["target_rank"].map(ndcg_at_5_from_rank).to_numpy(dtype=float)
d1_saved = pd.to_numeric(batch_d1_per_case["ndcg_at_5"], errors="raise").to_numpy(dtype=float)
if not np.isclose(d1_saved, d1_reconstructed, rtol=0.0, atol=FLOAT_TOLERANCE).all():
    raise RuntimeError("Batch D1 per-case NDCG@5 does not match reconstructed target_rank.")

require_columns(
    batch_d2_per_case,
    ["category_id", "case_id", "query_id", "user_id", "regime", "condition", "pool_depth", "target_rank", "ndcg_at_5"],
    "Batch D2 per-case metrics",
)
batch_d2_per_case = batch_d2_per_case.loc[
    batch_d2_per_case["category_id"].astype(str).eq(CATEGORY_ID)
    & pd.to_numeric(batch_d2_per_case["pool_depth"], errors="coerce").eq(BATCH_D2_POOL_DEPTH)
].copy()
require_unique(batch_d2_per_case, ["case_id", "condition"], "Batch D2 per-case metrics")
d2_reconstructed = batch_d2_per_case["target_rank"].map(ndcg_at_5_from_rank).to_numpy(dtype=float)
d2_saved = pd.to_numeric(batch_d2_per_case["ndcg_at_5"], errors="raise").to_numpy(dtype=float)
if not np.isclose(d2_saved, d2_reconstructed, rtol=0.0, atol=FLOAT_TOLERANCE).all():
    raise RuntimeError("Batch D2 per-case NDCG@5 does not match reconstructed target_rank.")

require_columns(
    batch_a_delta,
    [
        "category_id", "case_id", "query_id", "user_id", "regime", "contrast_name",
        "reranker_family", "candidate_pool_depth", "metric_name", "metric_cutoff",
        "metric_value_left", "metric_value_right", "target_rank_left", "target_rank_right",
        "delta_ndcg_at_5", "left_condition", "right_condition", "candidate_source_left",
        "candidate_source_right", "source_notebook_left", "source_notebook_right",
    ],
    "Batch A stage_delta_per_case",
)
batch_a_delta = batch_a_delta.loc[
    batch_a_delta["category_id"].astype(str).eq(CATEGORY_ID)
    & pd.to_numeric(batch_a_delta["candidate_pool_depth"], errors="coerce").eq(BATCH_A_POOL_DEPTH)
    & batch_a_delta["metric_name"].astype(str).str.upper().eq("NDCG")
    & pd.to_numeric(batch_a_delta["metric_cutoff"], errors="coerce").eq(PRIMARY_K)
    & batch_a_delta["contrast_name"].isin(REQUIRED_STAGE_CONTRASTS)
].copy()
if batch_a_delta.empty:
    raise RuntimeError(
        "Batch A has no category-specific NDCG@5 stage deltas at pool depth "
        f"{BATCH_A_POOL_DEPTH}."
    )
require_unique(batch_a_delta, ["case_id", "contrast_name", "reranker_family"], "Batch A stage deltas")
for contrast, (expected_left, expected_right) in CANONICAL_CONDITION_PAIRS.items():
    observed_conditions = batch_a_delta.loc[
        batch_a_delta["contrast_name"].eq(contrast), ["left_condition", "right_condition"]
    ].drop_duplicates()
    if len(observed_conditions) != 1 or tuple(observed_conditions.iloc[0]) != (expected_left, expected_right):
        raise RuntimeError(
            f"Canonical contrast contract changed for {contrast}: {observed_conditions.to_dict('records')}"
        )
if batch_a_delta[["candidate_source_left", "candidate_source_right", "source_notebook_left", "source_notebook_right"]].isna().any().any():
    raise RuntimeError("Batch A candidate-source or source-notebook lineage is missing.")
lineage_text = batch_a_delta[["candidate_source_left", "candidate_source_right", "source_notebook_left", "source_notebook_right"]].astype(str).apply(lambda column: column.str.strip().str.lower())
if lineage_text.apply(lambda column: column.isin({"", "nan", "none", "<na>"})).any().any():
    raise RuntimeError("Batch A candidate-source or source-notebook lineage contains blank/null-like values.")

expected_pairs = {
    ("P1-only_minus_P0", "shared_stage1"),
    *{(contrast, PRIMARY_RERANKER) for contrast in REQUIRED_STAGE_CONTRASTS[1:]},
}
observed_pairs = set(batch_a_delta[["contrast_name", "reranker_family"]].drop_duplicates().itertuples(index=False, name=None))
missing_pairs = expected_pairs.difference(observed_pairs)
if missing_pairs:
    raise RuntimeError(f"Batch A is missing required contrast/reranker pairs: {sorted(missing_pairs)}")
unexpected_transformer_contrasts = {
    contrast for contrast, family in observed_pairs
    if family == SECONDARY_RERANKER and contrast not in REQUIRED_STAGE_CONTRASTS[1:]
}
if unexpected_transformer_contrasts:
    raise RuntimeError(f"Unexpected Transformer contrast labels: {sorted(unexpected_transformer_contrasts)}")

# Confirm that Batch E consumes the authoritative target-rank reconstruction.
for side in ["left", "right"]:
    reconstructed = batch_a_delta[f"target_rank_{side}"].map(ndcg_at_5_from_rank).to_numpy(dtype=float)
    saved = pd.to_numeric(batch_a_delta[f"metric_value_{side}"], errors="raise").to_numpy(dtype=float)
    mismatch = ~np.isclose(saved, reconstructed, rtol=0.0, atol=FLOAT_TOLERANCE)
    if mismatch.any():
        raise RuntimeError(
            f"Batch A {side} NDCG@5 does not match authoritative target_rank for {int(mismatch.sum())} rows."
        )
calculated_delta = (
    pd.to_numeric(batch_a_delta["metric_value_right"], errors="raise")
    - pd.to_numeric(batch_a_delta["metric_value_left"], errors="raise")
)
if not np.isclose(
    calculated_delta.to_numpy(dtype=float),
    pd.to_numeric(batch_a_delta["delta_ndcg_at_5"], errors="raise").to_numpy(dtype=float),
    rtol=0.0,
    atol=FLOAT_TOLERANCE,
).all():
    raise RuntimeError("Batch A continuous delta orientation is not right minus left.")

for label, diagnostic_delta in [("Batch D1", batch_d1_delta), ("Batch D2", batch_d2_delta)]:
    require_columns(
        diagnostic_delta,
        ["left_ndcg_at_5", "right_ndcg_at_5", "delta_ndcg_at_5"],
        f"{label} paired deltas",
    )
    expected_delta = (
        pd.to_numeric(diagnostic_delta["left_ndcg_at_5"], errors="raise")
        - pd.to_numeric(diagnostic_delta["right_ndcg_at_5"], errors="raise")
    )
    if not np.isclose(
        expected_delta.to_numpy(dtype=float),
        pd.to_numeric(diagnostic_delta["delta_ndcg_at_5"], errors="raise").to_numpy(dtype=float),
        rtol=0.0,
        atol=FLOAT_TOLERANCE,
    ).all():
        raise RuntimeError(f"{label} controlled delta orientation is not left minus right.")

require_columns(batch_a_authority_qc, ["qc_check", "canonical_summary_uses_reconstructed_metric", "check_passed"], "Batch A metric authority QC")
authority_required = batch_a_authority_qc.loc[
    batch_a_authority_qc["qc_check"].isin(["canonical NDCG@5 reconstruction", "Notebook 14 summary reconciliation"])
]
if len(authority_required) != 2:
    raise RuntimeError("Batch A metric-authority QC is missing one of the two required authority checks.")
if not boolean_series(authority_required["canonical_summary_uses_reconstructed_metric"]).all():
    raise RuntimeError("Batch A reports a summary that does not use reconstructed target-rank metrics.")
if not boolean_series(authority_required["check_passed"]).all():
    raise RuntimeError("Batch A target-rank authority or Notebook 14 reconciliation QC failed.")

print("Batch A authoritative target-rank metric checks: passed")
print("Stage-delta rows:", len(batch_a_delta))


Batch A authoritative target-rank metric checks: passed
Stage-delta rows: 16016


In [42]:
# ==== Discover Standardized Batch B case-level Mechanism Features ====
BATCH_B_TOKENS = (
    "structurality", "preference_alignment", "alignment", "coherence",
    "item_evidence", "headroom", "brand_affinity", "mechanism_feature",
)
FEATURE_TOKENS = (
    "alignment", "coherence", "entropy", "dispersion", "concentration", "dominant",
    "history", "prior", "facet", "brand", "idf", "rare_token", "source_signal",
    "item_evidence", "coverage", "retention", "specific_family", "composite",
)
OUTCOME_TOKENS = (
    "ndcg", "mrr", "hit_at", "delta", "rerank_rank", "target_rank", "outcome_group",
    "prediction", "score_label",
)


def print_batch_b_resolution_diagnostics(discovery_rows, missing_rows, preferred_inputs):
    print("Batch B artifact resolution diagnostic")
    print("Preferred inputs:")
    for spec in preferred_inputs:
        print("-", spec["source_notebook"], "manifest=", spec["manifest_path"], "artifact=", spec["artifact_path"])
    print("Candidate manifest paths searched:")
    for candidate in BATCH_B_MANIFEST_HINTS:
        print("-", candidate)
    if missing_rows:
        print("Missing preferred artifacts or manifest fields:")
        print(pd.DataFrame(missing_rows).to_string(index=False))
    if discovery_rows:
        diagnostic_df = pd.DataFrame(discovery_rows)
        visible_columns = [
            column for column in [
                "artifact_path", "row_count", "column_count", "id_available",
                "candidate_feature_count", "accepted_as_batch_b_case_feature_input",
                "observed_columns_sample",
            ] if column in diagnostic_df.columns
        ]
        print("Observed candidate schemas, first 20 rows:")
        print(diagnostic_df[visible_columns].head(20).to_string(index=False))
    print("Evidence needed: the listed Face Batch B manifests plus their case-level feature tables with case_id or query_id and the required role columns.")


preferred_by_artifact = {Path(spec["artifact_path"]): spec for spec in BATCH_B_PREFERRED_INPUTS}
manifest_candidates = {path for path in BATCH_B_MANIFEST_HINTS if path.exists()}
preferred_missing_rows = []
for spec in BATCH_B_PREFERRED_INPUTS:
    manifest_path = Path(spec["manifest_path"])
    artifact_path = Path(spec["artifact_path"])
    if manifest_path.exists():
        manifest_candidates.add(manifest_path)
    else:
        preferred_missing_rows.append({
            "source_notebook": spec["source_notebook"],
            "missing_path": str(manifest_path),
            "missing_type": "manifest",
            "required_manifest_fields": "category_id, output_paths/input_paths, validation/history contract where present",
        })
    if not artifact_path.exists():
        preferred_missing_rows.append({
            "source_notebook": spec["source_notebook"],
            "missing_path": str(artifact_path),
            "missing_type": "case_feature_artifact",
            "required_columns_or_roles": json.dumps(spec["required_role_columns"], ensure_ascii=False),
        })

for path in ANALYSIS_DIR.rglob("*manifest*.json"):
    path_text = normalized_name(path)
    if any(token in path_text for token in BATCH_B_TOKENS):
        manifest_candidates.add(path)
for path in ANALYSIS_DIR.rglob("run_manifest.json"):
    path_text = normalized_name(path)
    if any(token in path_text for token in BATCH_B_TOKENS):
        manifest_candidates.add(path)

batch_b_manifests = []
batch_b_artifact_candidates = set()
for manifest_path in sorted(manifest_candidates):
    payload = load_json(manifest_path)
    declared_category = payload.get("category_id")
    if declared_category is not None and str(declared_category) != CATEGORY_ID:
        continue
    searchable = normalized_name(manifest_path) + " " + normalized_name(json.dumps(payload, default=str))
    if not any(token in searchable for token in BATCH_B_TOKENS):
        continue
    batch_b_manifests.append((manifest_path, payload))
    for artifact_path in recursive_paths(payload):
        resolved = artifact_path if artifact_path.is_absolute() else PROJECT_ROOT / artifact_path
        if resolved.exists():
            batch_b_artifact_candidates.add(resolved)
    for artifact_path in manifest_path.parent.iterdir():
        if artifact_path.suffix.lower() in {".parquet", ".csv"}:
            batch_b_artifact_candidates.add(artifact_path)
for spec in BATCH_B_PREFERRED_INPUTS:
    artifact_path = Path(spec["artifact_path"])
    if artifact_path.exists():
        batch_b_artifact_candidates.add(artifact_path)

batch_b_frames = []
batch_b_discovery_rows = []
batch_b_role_qc_rows = []
for artifact_path in sorted(batch_b_artifact_candidates):
    file_name = normalized_name(artifact_path.name)
    if not any(token in file_name for token in ("case", "feature", "alignment", "structurality", "preference", "coherence", "brand")):
        continue
    if any(token in file_name for token in ("candidate", "prediction", "importance", "summary", "qc")):
        continue
    frame = read_table(artifact_path)
    id_available = "case_id" in frame.columns or "query_id" in frame.columns
    feature_columns = [
        column for column in frame.columns
        if any(token in normalized_name(column) for token in FEATURE_TOKENS)
        and not any(token in normalized_name(column) for token in OUTCOME_TOKENS)
    ]
    preferred_spec = preferred_by_artifact.get(Path(artifact_path))
    if preferred_spec is not None:
        if frame.empty:
            preferred_missing_rows.append({
                "source_notebook": preferred_spec["source_notebook"],
                "missing_path": str(artifact_path),
                "missing_type": "empty_case_feature_artifact",
                "required_columns_or_roles": json.dumps(preferred_spec["required_role_columns"], ensure_ascii=False),
            })
        for role, columns in preferred_spec["required_role_columns"].items():
            present = [column for column in columns if column in frame.columns]
            populated = int(frame[present].notna().any(axis=1).sum()) if present else 0
            batch_b_role_qc_rows.append({
                "artifact_path": str(artifact_path),
                "source_notebook": preferred_spec["source_notebook"],
                "feature_role": role,
                "required_any_columns": columns,
                "present_columns": present,
                "populated_case_count": populated,
                "passed": bool(present and populated > 0),
            })
            if not present or populated == 0:
                preferred_missing_rows.append({
                    "source_notebook": preferred_spec["source_notebook"],
                    "missing_path": str(artifact_path),
                    "missing_type": f"missing_or_empty_role::{role}",
                    "required_columns_or_roles": json.dumps(columns, ensure_ascii=False),
                    "observed_columns_sample": json.dumps(list(frame.columns[:50]), ensure_ascii=False),
                })
    accepted = bool(id_available and feature_columns)
    batch_b_discovery_rows.append({
        "artifact_path": str(artifact_path),
        "row_count": int(len(frame)),
        "column_count": int(len(frame.columns)),
        "id_available": id_available,
        "candidate_feature_count": len(feature_columns),
        "accepted_as_batch_b_case_feature_input": accepted,
        "observed_columns_sample": list(frame.columns[:30]),
    })
    if accepted:
        batch_b_frames.append((artifact_path, frame, feature_columns))

resolved_batch_b_artifact_paths = {
    Path(artifact_path) for artifact_path, _, _ in batch_b_frames
}
notebook24_resolved = Path(INPUT_PATHS["preference_alignment_features"]) in resolved_batch_b_artifact_paths

legacy_structurality_resolved = any(
    "structurality" in normalized_name(artifact_path)
    and "outdated" in normalized_name(artifact_path)
    for artifact_path, _, _ in batch_b_frames
)

fatal_batch_b_missing = (
    not batch_b_manifests
    or not batch_b_frames
    or not notebook24_resolved
)

if fatal_batch_b_missing:
    print_batch_b_resolution_diagnostics(
        batch_b_discovery_rows,
        preferred_missing_rows,
        BATCH_B_PREFERRED_INPUTS,
    )
    raise FileNotFoundError(
        "No usable Batch B per-case mechanism feature inputs were resolved. "
        "Notebook 24 case_alignment_features.parquet is required; structurality may be a discovered legacy case-feature artifact."
    )

if preferred_missing_rows:
    print_batch_b_resolution_diagnostics(
        batch_b_discovery_rows,
        preferred_missing_rows,
        BATCH_B_PREFERRED_INPUTS,
    )
    print(
        "WARNING: Preferred standardized Batch B structurality outputs are incomplete. "
        "Continuing with discovered case-level Batch B feature artifacts. "
        f"Legacy structurality resolved: {legacy_structurality_resolved}"
    )
batch_b_discovery_qc = pd.DataFrame(batch_b_discovery_rows)
batch_b_role_qc = pd.DataFrame(batch_b_role_qc_rows)
print("Batch B manifests:", [str(path) for path, _ in batch_b_manifests])
print("Accepted Batch B case-feature artifacts:", [str(path) for path, _, _ in batch_b_frames])
display(batch_b_role_qc)


Batch B artifact resolution diagnostic
Preferred inputs:
- 23_structurality_analysis_face.ipynb manifest= /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/structurality_analysis/run_manifest.json artifact= /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/structurality_analysis/preference_structurality_features_by_case.csv
- 24_preference_alignment_face.ipynb manifest= /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment/alignment_definition_manifest.json artifact= /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment/case_alignment_features.parquet
Candidate manifest paths searched:
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/structurality_analysis/run_manifest.json
- /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment/alignment_definition_

,artifact_path,source_notebook,feature_role,required_any_columns,present_columns,populated_case_count,passed
0,/content/drive/MyDrive/thesis_recsys/categorie...,24_preference_alignment_face.ipynb,query_history_coherence,"[query_profile_functional_composite, qchs_alig...","[query_profile_functional_composite, qchs_alig...",2288,True
1,/content/drive/MyDrive/thesis_recsys/categorie...,24_preference_alignment_face.ipynb,history_quantity,"[strict_prior_interaction_count, strict_prior_...","[strict_prior_interaction_count, strict_prior_...",2288,True
2,/content/drive/MyDrive/thesis_recsys/categorie...,24_preference_alignment_face.ipynb,baseline_item_evidence,"[query_target_functional_composite, query_targ...","[query_target_functional_composite, query_targ...",2288,True
3,/content/drive/MyDrive/thesis_recsys/categorie...,24_preference_alignment_face.ipynb,brand_coherence_or_affinity,"[target_brand_seen_in_prior, target_brand_prio...","[target_brand_seen_in_prior, target_brand_prio...",2288,True


In [43]:
# ==== Authoritative Case Crosswalk and join-grain Repair ====
case_crosswalk = batch_a_delta[["case_id", "query_id", "user_id", "regime"]].copy()
for column in ["case_id", "query_id", "user_id", "regime"]:
    case_crosswalk[column] = case_crosswalk[column].astype(str)
consistency = case_crosswalk.groupby("case_id", observed=True)[["query_id", "user_id", "regime"]].nunique(dropna=False)
if consistency.gt(1).any().any():
    raise RuntimeError("Batch A maps a case_id to multiple query_id, user_id, or regime values.")
case_crosswalk = case_crosswalk.drop_duplicates("case_id").reset_index(drop=True)
require_unique(case_crosswalk, ["case_id"], "Batch A case crosswalk")
require_unique(case_crosswalk, ["query_id"], "Batch A query crosswalk")
notebook24_feature_frame = select_notebook24_feature_frame(
    batch_b_frames,
    INPUT_PATHS["preference_alignment_features"],
)
notebook24_reader_contract = validate_notebook24_reader_contract(
    manifest=preference_alignment_manifest,
    manifest_path=INPUT_PATHS["preference_alignment_manifest"],
    feature_frame=notebook24_feature_frame,
    feature_path=INPUT_PATHS["preference_alignment_features"],
    case_crosswalk=case_crosswalk,
    category_id=CATEGORY_ID,
)
print("Notebook 24 reader-side contract: PASS")


def _key_text(series):
    return series.fillna("").astype(str).str.strip()


identifier_alias = pd.concat([
    case_crosswalk[["case_id", "query_id"]].rename(columns={"query_id": "identifier_alias"}),
    case_crosswalk[["case_id"]].assign(identifier_alias=case_crosswalk["case_id"]),
], ignore_index=True).drop_duplicates()
identifier_alias["identifier_alias"] = _key_text(identifier_alias["identifier_alias"])
identifier_alias["case_id"] = _key_text(identifier_alias["case_id"])
identifier_alias = identifier_alias.loc[identifier_alias["identifier_alias"].ne("")].copy()


def validate_identifier_alias(label):
    conflicts = identifier_alias.groupby("identifier_alias")["case_id"].nunique()
    if conflicts.gt(1).any():
        bad = conflicts.loc[conflicts.gt(1)].head(10).index.tolist()
        raise RuntimeError(f"{label} identifier alias map is ambiguous: {bad}")


validate_identifier_alias("base case_id/query_id")

require_columns(
    notebook24_feature_frame,
    ["case_id", "user_id", "regime", "target_parent_asin"],
    "Notebook 24 canonical target-key features",
)
canonical_target_key = notebook24_feature_frame[
    ["case_id", "user_id", "regime", "target_parent_asin"]
].copy()
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    canonical_target_key[column] = _key_text(canonical_target_key[column])
canonical_target_key["regime"] = canonical_target_key["regime"].str.lower()

canonical_target_key = canonical_target_key.merge(
    case_crosswalk[["case_id", "query_id"]].rename(columns={"query_id": "canonical_query_id"}),
    on="case_id",
    how="left",
    validate="one_to_one",
)
target_key_columns = ["user_id", "regime", "target_parent_asin"]
if canonical_target_key.duplicated(target_key_columns).any():
    sample = canonical_target_key.loc[
        canonical_target_key.duplicated(target_key_columns, keep=False),
        target_key_columns + ["case_id"],
    ].head(10).to_dict("records")
    raise RuntimeError(f"Notebook 24 target-key crosswalk is not one-to-one: {sample}")


def register_target_key_aliases(frame, label, minimum_coverage=1.0):
    global identifier_alias

    required = ["case_id", "user_id", "regime", "target_parent_asin"]
    if not set(required).issubset(frame.columns):
        return

    work_columns = required + (["query_id"] if "query_id" in frame.columns else [])
    work = frame[work_columns].drop_duplicates().copy()
    for column in work_columns:
        work[column] = _key_text(work[column])
    work["regime"] = work["regime"].str.lower()

    merged = work.merge(
        canonical_target_key[target_key_columns + ["case_id"]].rename(columns={"case_id": "_canonical_case_id"}),
        on=target_key_columns,
        how="left",
        validate="many_to_one",
    )
    matched = merged["_canonical_case_id"].notna()
    coverage = float(matched.mean()) if len(merged) else 0.0
    if coverage < minimum_coverage:
        raise RuntimeError(
            f"{label} target-key coverage {coverage:.3f} is below {minimum_coverage:.3f}; "
            "cannot repair stale case_id/query_id aliases."
        )

    alias_frames = []
    for source_column in ["case_id", "query_id"]:
        if source_column in merged.columns:
            alias_frames.append(
                merged.loc[matched, [source_column, "_canonical_case_id"]]
                .rename(columns={source_column: "identifier_alias", "_canonical_case_id": "case_id"})
            )

    if alias_frames:
        identifier_alias = pd.concat([identifier_alias, *alias_frames], ignore_index=True).drop_duplicates()
        identifier_alias["identifier_alias"] = _key_text(identifier_alias["identifier_alias"])
        identifier_alias["case_id"] = _key_text(identifier_alias["case_id"])
        identifier_alias = identifier_alias.loc[identifier_alias["identifier_alias"].ne("")].drop_duplicates()
        validate_identifier_alias(label)

    join_qc_rows.append({
        "input_name": f"{label} stale-id target-key alias repair",
        "input_rows": int(len(work)),
        "matched_rows": int(matched.sum()),
        "unmatched_rows": int((~matched).sum()),
        "coverage_rate": coverage,
        "key_source": "user_id+regime+target_parent_asin",
        "query_id_mismatch_count": 0,
        "user_id_mismatch_count": 0,
        "regime_mismatch_count": 0,
        "row_multiplication_count": 0,
        "status": "pass",
    })


join_qc_rows = []

register_target_key_aliases(batch_d1_profiles, "Batch D1 profile diagnostics", 1.0)
register_target_key_aliases(batch_d2_profiles, "Batch D2 profile diagnostics", 1.0)

def standardize_case_id(frame, label, minimum_coverage=1.0):
    work = frame.copy()
    input_rows = len(work)
    key_columns = [c for c in ["case_id", "query_id"] if c in work.columns]
    if not key_columns:
        raise RuntimeError(f"{label} has neither case_id nor query_id.")

    alias = identifier_alias.copy()
    alias["identifier_alias"] = alias["identifier_alias"].astype(str)
    alias["case_id"] = alias["case_id"].astype(str)
    alias_to_case = alias.set_index("identifier_alias")["case_id"]

    attempts = []
    for key_column in key_columns:
        candidate = work.copy()
        for column in ["case_id", "query_id", "user_id", "regime"]:
            if column in candidate.columns:
                candidate[f"_input_{column}"] = candidate[column].astype(str)

        candidate["_identifier_alias"] = candidate[key_column].astype(str)
        candidate = candidate.drop(columns=["case_id", "query_id", "user_id", "regime"], errors="ignore")
        merged = candidate.merge(
            alias,
            left_on="_identifier_alias",
            right_on="identifier_alias",
            how="left",
            validate="many_to_one",
            indicator=True,
        ).merge(
            case_crosswalk.rename(columns={
                "query_id": "_canonical_query_id",
                "user_id": "_canonical_user_id",
                "regime": "_canonical_regime",
            }),
            on="case_id",
            how="left",
            validate="many_to_one",
        )
        matched = merged["_merge"].eq("both") & merged["_canonical_query_id"].notna()
        coverage = float(matched.mean()) if len(merged) else 0.0
        attempts.append((coverage, key_column, merged, matched))

    coverage, key_column, merged, matched = max(attempts, key=lambda item: item[0])
    if coverage < minimum_coverage:
        coverage_report = {key: round(cov, 6) for cov, key, _, _ in attempts}
        raise RuntimeError(
            f"{label} case-key coverage {coverage:.3f} is below {minimum_coverage:.3f}; "
            f"attempted_key_coverages={coverage_report}"
        )

    merged = merged.loc[matched].copy()

    def alias_conflict_count(input_column):
        column = f"_input_{input_column}"
        if column not in merged.columns:
            return 0
        mapped = merged[column].astype(str).map(alias_to_case)
        return int((mapped.notna() & mapped.ne(merged["case_id"].astype(str))).sum())

    query_mismatch = alias_conflict_count("query_id")
    case_id_mismatch = alias_conflict_count("case_id")
    if case_id_mismatch or query_mismatch:
        raise RuntimeError(
            f"{label} identifier aliases conflict with canonical case_id: "
            f"case_id={case_id_mismatch}, query_id={query_mismatch}"
        )

    user_mismatch = 0
    if "_input_user_id" in merged.columns:
        user_mismatch = int((~merged["_input_user_id"].astype(str).eq(merged["_canonical_user_id"].astype(str))).sum())
        if user_mismatch:
            raise RuntimeError(f"{label} has {user_mismatch} user_id mismatches.")

    regime_mismatch = 0
    if "_input_regime" in merged.columns:
        regime_mismatch = int((~merged["_input_regime"].astype(str).str.lower().eq(merged["_canonical_regime"].astype(str).str.lower())).sum())
        if regime_mismatch:
            raise RuntimeError(f"{label} has {regime_mismatch} regime mismatches.")

    merged["query_id"] = merged["_canonical_query_id"].astype(str)
    merged["user_id"] = merged["_canonical_user_id"].astype(str)
    merged["regime"] = merged["_canonical_regime"].astype(str).str.lower()
    merged = merged.drop(
        columns=[
            "_merge", "identifier_alias", "_identifier_alias",
            "_canonical_query_id", "_canonical_user_id", "_canonical_regime",
            *[c for c in merged.columns if c.startswith("_input_")],
        ],
        errors="ignore",
    )

    join_qc_rows.append({
        "input_name": label,
        "input_rows": input_rows,
        "matched_rows": int(len(merged)),
        "unmatched_rows": int(input_rows - len(merged)),
        "coverage_rate": coverage,
        "key_source": f"{key_column}_alias",
        "query_id_mismatch_count": query_mismatch,
        "user_id_mismatch_count": user_mismatch,
        "regime_mismatch_count": regime_mismatch,
        "row_multiplication_count": 0,
        "status": "pass",
    })
    return merged


batch_d1_per_case = standardize_case_id(batch_d1_per_case, "Batch D1 per-case metrics", 1.0)
batch_d2_per_case = standardize_case_id(batch_d2_per_case, "Batch D2 per-case metrics", 1.0)
batch_d1_profiles = standardize_case_id(batch_d1_profiles, "Batch D1 profile diagnostics", 1.0)
batch_d1_delta = standardize_case_id(batch_d1_delta, "Batch D1 paired deltas", 1.0)
batch_d2_profiles = standardize_case_id(batch_d2_profiles, "Batch D2 profile diagnostics", 1.0)
batch_d2_delta = standardize_case_id(batch_d2_delta, "Batch D2 paired deltas", 1.0)
require_unique(batch_d1_per_case, ["case_id", "prior_policy"], "Batch D1 per-case metrics after crosswalk")
require_unique(batch_d2_per_case, ["case_id", "condition"], "Batch D2 per-case metrics after crosswalk")
require_unique(batch_d1_profiles, ["case_id"], "Batch D1 profile diagnostics")
require_unique(batch_d2_profiles, ["case_id"], "Batch D2 profile diagnostics")

case_features = case_crosswalk.copy()


def merge_prefixed_case_features(base, frame, prefix, columns):
    selected = frame[["case_id"] + columns].copy()
    require_unique(selected, ["case_id"], f"{prefix} selected case features")
    renamed = {column: f"{prefix}__{column}" for column in columns}
    before = len(base)
    merged = base.merge(selected.rename(columns=renamed), on="case_id", how="left", validate="one_to_one")
    if len(merged) != before:
        raise RuntimeError(f"{prefix} feature join multiplied or dropped rows.")
    return merged


d1_feature_columns = [column for column in batch_d1_profiles.columns if column not in {"case_id", "query_id", "user_id", "regime", "target_parent_asin"}]
d2_feature_columns = [column for column in batch_d2_profiles.columns if column not in {"case_id", "query_id", "user_id", "regime", "target_parent_asin"}]
case_features = merge_prefixed_case_features(case_features, batch_d1_profiles, "d1", d1_feature_columns)
case_features = merge_prefixed_case_features(case_features, batch_d2_profiles, "d2", d2_feature_columns)

batch_b_feature_sources = []
for artifact_path, frame, candidate_columns in batch_b_frames:
    standardized = standardize_case_id(frame, f"Batch B {artifact_path.name}", 0.01)
    if standardized.duplicated("case_id").any():
        varying = standardized.groupby("case_id", observed=True)[candidate_columns].nunique(dropna=False).gt(1).any(axis=1)
        if varying.any():
            join_qc_rows.append({
                "input_name": f"Batch B {artifact_path.name} case-grain check", "input_rows": len(standardized),
                "matched_rows": 0, "unmatched_rows": len(standardized), "coverage_rate": 0.0,
                "key_source": "case_id", "query_id_mismatch_count": 0, "user_id_mismatch_count": 0,
                "regime_mismatch_count": 0, "row_multiplication_count": int(standardized.duplicated("case_id", keep=False).sum()),
                "status": "excluded_non_case_grain",
            })
            continue
        standardized = standardized.drop_duplicates("case_id")

    numeric_features = []
    for column in candidate_columns:
        converted = pd.to_numeric(standardized[column], errors="coerce")
        if converted.notna().sum() >= max(1, int(0.5 * standardized[column].notna().sum())):
            standardized[column] = converted
            numeric_features.append(column)
    if not numeric_features:
        continue
    prefix = "b_" + normalized_name(artifact_path.stem)[:60]
    case_features = merge_prefixed_case_features(case_features, standardized, prefix, numeric_features)
    batch_b_feature_sources.append({
        "artifact_path": str(artifact_path), "prefix": prefix,
        "feature_columns": numeric_features, "case_count": int(standardized["case_id"].nunique()),
    })

if not batch_b_feature_sources:
    raise RuntimeError("Batch B artifacts were found, but none supplied a safe one-row-per-case numeric feature table.")

require_unique(case_features, ["case_id"], "Combined mechanism feature table")
print("Authoritative cases:", len(case_crosswalk))
print("Combined feature columns:", len(case_features.columns))


Notebook 24 reader-side contract: PASS
Authoritative cases: 2288
Combined feature columns: 164


In [44]:
# ==== Mechanism Feature Contract and continuous-delta Master Tables ====
def find_feature(aliases, required=False, preferred_prefix=None):
    candidates = [column for column in case_features.columns if column not in {"case_id", "query_id", "user_id", "regime"}]
    for alias in aliases:
        normalized_alias = normalized_name(alias)
        matches = [column for column in candidates if normalized_name(column).endswith(normalized_alias)]
        if matches:
            if preferred_prefix:
                preferred = [column for column in matches if column.startswith(preferred_prefix)]
                if preferred:
                    matches = preferred
            matches = sorted(matches, key=lambda column: (-case_features[column].notna().sum(), column))
            selected = matches[0]
            for alternative in matches[1:]:
                overlap = case_features[selected].notna() & case_features[alternative].notna()
                if not overlap.any():
                    raise RuntimeError(
                        f"Ambiguous disjoint mechanism features for alias {alias}: {selected}, {alternative}"
                    )
                left = pd.to_numeric(case_features.loc[overlap, selected], errors="coerce")
                right = pd.to_numeric(case_features.loc[overlap, alternative], errors="coerce")
                numeric_overlap = left.notna() & right.notna()
                if numeric_overlap.any():
                    equal = np.isclose(
                        left.loc[numeric_overlap].to_numpy(dtype=float),
                        right.loc[numeric_overlap].to_numpy(dtype=float),
                        rtol=0.0, atol=FLOAT_TOLERANCE,
                    ).all()
                else:
                    equal = case_features.loc[overlap, selected].astype(str).eq(
                        case_features.loc[overlap, alternative].astype(str)
                    ).all()
                if not equal:
                    raise RuntimeError(
                        f"Conflicting mechanism features for alias {alias}: {selected}, {alternative}"
                    )
            return selected
    if required:
        raise RuntimeError(f"No mechanism feature matches aliases: {aliases}")
    return None


COHERENCE_ALIASES = [
    "query_profile_functional_alignment", "query_history_functional_coherence",
    "query_history_coherence", "profile_query_alignment", "qchs_alignment_mean",
]
HISTORY_QUANTITY_ALIASES = [
    "strict_prior_interaction_count", "strict_prior_unique_item_count",
    "unique_prior_item_count", "qchs_input_prior_interaction_count",
    "qchs_input_prior_unique_item_count", "eligible_prior_event_count",
    "eligible_prior_unique_item_count", "prior_interaction_count",
]
BRAND_ALIASES = [
    "brand_coherence", "query_profile_brand_alignment", "target_profile_brand_alignment",
    "dominant_brand_share", "profile_brand_concentration", "brand_entropy",
    "unique_prior_brand_count",
]
ITEM_EVIDENCE_ALIASES = [
    "baseline_item_evidence", "query_target_functional_alignment", "query_specific_family_count",
    "source_signal_count", "catalog_relative_idf", "rare_token_share",
]

coherence_feature = find_feature(COHERENCE_ALIASES, required=True, preferred_prefix="b_")
history_quantity_feature = find_feature(HISTORY_QUANTITY_ALIASES, required=True, preferred_prefix="b_")
brand_features = []
for alias in BRAND_ALIASES:
    feature = find_feature(
        [alias],
        required=False,
        preferred_prefix="b_case_alignment_features__",
    )
    if feature and feature not in brand_features:
        brand_features.append(feature)
brand_features = brand_features[:4]
item_evidence_features = []
for alias in ITEM_EVIDENCE_ALIASES:
    feature = find_feature([alias], required=False, preferred_prefix="b_")
    if feature and feature not in item_evidence_features:
        item_evidence_features.append(feature)
item_evidence_features = item_evidence_features[:6]

stage1_qchs_active_feature = find_feature(["stage1_qchs_active", "qchs_active"], required=True, preferred_prefix="d1__")
stage2_qchs_active_feature = find_feature(["stage2_qchs_filtered_prior_active", "qchs_active"], required=True, preferred_prefix="d2__")
case_features["stage1_qchs_active"] = boolean_series(case_features[stage1_qchs_active_feature])
case_features["stage2_qchs_filtered_prior_active"] = boolean_series(case_features[stage2_qchs_active_feature])
case_features["stage1_qchs_status"] = np.where(case_features["stage1_qchs_active"], "Stage1-QCHS-active", "Stage1-QCHS-fallback")
case_features["stage2_qchs_filtered_prior_status"] = np.where(
    case_features["stage2_qchs_filtered_prior_active"],
    "Stage2-QCHS-filtered-prior-active",
    "Stage2-QCHS-filtered-prior-inactive",
)
case_features["is_strong"] = case_features["regime"].str.lower().eq("strong")

feature_contract_rows = [
    {
        "analysis_role": "query_history_coherence", "selected_column": coherence_feature,
        "source_batch": "D1" if coherence_feature.startswith("d1__") else "B",
        "availability": "available", "selection_rule": "alias priority, maximum coverage, and equality check across duplicate aliases",
        "causal_interpretation_allowed": False,
    },
    {
        "analysis_role": "history_quantity", "selected_column": history_quantity_feature,
        "source_batch": "D1" if history_quantity_feature.startswith("d1__") else "B",
        "availability": "available", "selection_rule": "alias priority, maximum coverage, and equality check across duplicate aliases",
        "causal_interpretation_allowed": False,
    },
    {
        "analysis_role": "stage1_qchs_active_status", "selected_column": stage1_qchs_active_feature,
        "source_batch": "D1",
        "availability": "available", "selection_rule": "controlled diagnostic status",
        "causal_interpretation_allowed": False,
    },
    {
        "analysis_role": "stage2_qchs_filtered_prior_active_status", "selected_column": stage2_qchs_active_feature,
        "source_batch": "D2",
        "availability": "available", "selection_rule": "controlled Stage-2 prior-policy diagnostic status",
        "causal_interpretation_allowed": False,
    },
]
for feature in brand_features:
    feature_contract_rows.append({
        "analysis_role": "brand_coherence_heterogeneity", "selected_column": feature,
        "source_batch": "B" if feature.startswith("b_") else "D1/D2",
        "availability": "available", "selection_rule": "documented brand feature alias",
        "causal_interpretation_allowed": False,
    })
for feature in item_evidence_features:
    feature_contract_rows.append({
        "analysis_role": "baseline_item_evidence", "selected_column": feature,
        "source_batch": "B", "availability": "available",
        "selection_rule": "documented item-evidence feature alias", "causal_interpretation_allowed": False,
    })
if not brand_features:
    feature_contract_rows.append({
        "analysis_role": "brand_coherence_heterogeneity", "selected_column": "",
        "source_batch": "B", "availability": "unavailable",
        "selection_rule": "No standardized Batch B brand-coherence column resolved",
        "causal_interpretation_allowed": False,
    })
feature_contract = pd.DataFrame(feature_contract_rows)

stage_master = batch_a_delta.merge(case_features, on="case_id", how="left", suffixes=("", "_feature"), validate="many_to_one")
if len(stage_master) != len(batch_a_delta):
    raise RuntimeError("Case-feature join multiplied or dropped Batch A stage-delta rows.")
for column in ["query_id", "user_id", "regime"]:
    feature_column = f"{column}_feature"
    if feature_column in stage_master.columns:
        mismatch = ~stage_master[column].astype(str).str.lower().eq(stage_master[feature_column].astype(str).str.lower())
        if mismatch.any():
            raise RuntimeError(f"Batch A and feature table disagree on {column}.")
        stage_master = stage_master.drop(columns=[feature_column])
stage_master["analysis_source"] = "Batch A canonical stage deltas"
stage_master["effect_label"] = stage_master["contrast_name"].map(STAGE_EFFECT_LABELS)
stage_master["baseline_headroom_ndcg_at_5"] = 1.0 - pd.to_numeric(stage_master["metric_value_left"], errors="raise")
stage_master["baseline_target_rank"] = pd.to_numeric(stage_master["target_rank_left"], errors="coerce")
stage_master["target_rank_improvement_descriptive"] = (
    pd.to_numeric(stage_master["target_rank_left"], errors="coerce")
    - pd.to_numeric(stage_master["target_rank_right"], errors="coerce")
)
stage_master["population_overall"] = True
stage_master["population_strong"] = stage_master["regime"].astype(str).str.lower().eq("strong")

require_columns(batch_d1_delta, ["contrast", "delta_ndcg_at_5", "case_id", "user_id", "regime", "pool_depth"], "Batch D1 paired deltas")
d1_qchs_all = batch_d1_delta.loc[
    batch_d1_delta["contrast"].eq("QCHS_minus_All_Prior")
    & pd.to_numeric(batch_d1_delta["pool_depth"],
    errors="coerce").eq(BATCH_D1_POOL_DEPTH)
].copy()
require_unique(d1_qchs_all, ["case_id", "contrast"], "Batch D1 QCHS versus All Prior")
d1_qchs_all = d1_qchs_all.merge(
    case_features,
    on="case_id",
    how="left",
    suffixes=("", "_feature"),
    validate="one_to_one",
)
d1_qchs_all["contrast_name"] = "QCHS_minus_All_Prior"
d1_qchs_all["reranker_family"] = "stage1_controlled_retrieval"
d1_qchs_all["analysis_source"] = "Batch D1"

require_columns(
    batch_d2_delta,
    ["contrast", "delta_ndcg_at_5", "case_id", "user_id", "regime", "pool_depth"],
    "Batch D2 paired deltas",
)
d2_diagnostic = batch_d2_delta.loc[
    batch_d2_delta["contrast"].isin(["QCHS_minus_Actual", "Actual_minus_No_User_Brand"])
    & pd.to_numeric(batch_d2_delta["pool_depth"],
    errors="coerce").eq(BATCH_D2_POOL_DEPTH)
].copy()
require_unique(d2_diagnostic, ["case_id", "contrast"], "Batch D2 diagnostic deltas")
d2_diagnostic = d2_diagnostic.merge(
    case_features,
    on="case_id",
    how="left",
    suffixes=("", "_feature"),
    validate="many_to_one",
)
d2_diagnostic["contrast_name"] = d2_diagnostic["contrast"].map({
    "QCHS_minus_Actual": "QCHS_Filtered_minus_Actual_All_Prior",
    "Actual_minus_No_User_Brand": "Actual_All_Prior_minus_No_User_Brand",
})
d2_diagnostic["reranker_family"] = "lightgbm_controlled"
d2_diagnostic["analysis_source"] = "Batch D2"

display(feature_contract)
display(stage_master[["contrast_name", "reranker_family", "regime", "delta_ndcg_at_5", "baseline_headroom_ndcg_at_5"]].head())


,analysis_role,selected_column,source_batch,availability,selection_rule,causal_interpretation_allowed
0,query_history_coherence,b_preference_structurality_features_by_case__q...,B,available,"alias priority, maximum coverage, and equality...",False
1,history_quantity,b_case_alignment_features__strict_prior_intera...,B,available,"alias priority, maximum coverage, and equality...",False
2,stage1_qchs_active_status,d1__stage1_qchs_active,D1,available,controlled diagnostic status,False
3,stage2_qchs_filtered_prior_active_status,d2__stage2_qchs_filtered_prior_active,D2,available,controlled Stage-2 prior-policy diagnostic status,False
4,brand_coherence_heterogeneity,b_case_alignment_features__dominant_brand_share,B,available,documented brand feature alias,False
5,brand_coherence_heterogeneity,b_preference_structurality_features_by_case__b...,B,available,documented brand feature alias,False
6,brand_coherence_heterogeneity,b_case_alignment_features__unique_prior_brand_...,B,available,documented brand feature alias,False
7,baseline_item_evidence,b_preference_structurality_features_by_case__q...,B,available,documented item-evidence feature alias,False
8,baseline_item_evidence,b_preference_structurality_features_by_case__q...,B,available,documented item-evidence feature alias,False
9,baseline_item_evidence,b_case_alignment_features__query_source_signal...,B,available,documented item-evidence feature alias,False


,contrast_name,reranker_family,regime,delta_ndcg_at_5,baseline_headroom_ndcg_at_5
0,P1-only_minus_P0,shared_stage1,cold,0.0,1.0
1,P1-only_minus_P0,shared_stage1,cold,0.0,1.0
2,P1-only_minus_P0,shared_stage1,cold,0.0,1.0
3,P1-only_minus_P0,shared_stage1,cold,0.0,1.0
4,P1-only_minus_P0,shared_stage1,cold,0.0,1.0


## Results

In [45]:
# ==== Adaptive Stratification Helpers ====
stratification_qc_rows = []


def adaptive_strata(frame, feature, analysis_name, population, min_cell_n=MIN_STRATUM_N):
    values = pd.to_numeric(frame[feature], errors="coerce")
    valid = values.notna() & np.isfinite(values)
    labels = pd.Series(pd.NA, index=frame.index, dtype="string")
    method = "unavailable"
    reason = "insufficient non-missing observations"
    if valid.sum() >= 4 * min_cell_n and values.loc[valid].nunique() >= 4:
        try:
            bins = pd.qcut(values.loc[valid], q=4, duplicates="drop")
            categories = list(bins.cat.categories)
            counts = bins.value_counts(sort=False)
            if len(categories) == 4 and int(counts.min()) >= min_cell_n:
                mapping = {interval: f"Q{index + 1}" for index, interval in enumerate(categories)}
                labels.loc[valid] = bins.map(mapping).astype("string")
                method = "quartiles"
                reason = "four qcut cells satisfy the minimum size"
        except ValueError:
            pass
    if method == "unavailable" and valid.sum() >= 2 * min_cell_n and values.loc[valid].nunique() >= 2:
        median = float(values.loc[valid].median())
        proposed = pd.Series(np.where(values.loc[valid].le(median), "low", "high"), index=values.loc[valid].index, dtype="string")
        counts = proposed.value_counts()
        if set(counts.index) == {"low", "high"} and int(counts.min()) >= min_cell_n:
            labels.loc[valid] = proposed
            method = "median_high_low"
            reason = f"quartile cells were insufficient; split at median={median:.8g}"
        else:
            reason = "quartiles and median high/low both fail minimum cell size"
    stratification_qc_rows.append({
        "analysis_name": analysis_name, "population": population, "feature": feature,
        "available_case_count": int(valid.sum()), "unique_value_count": int(values.loc[valid].nunique()),
        "method": method, "minimum_cell_n": min_cell_n,
        "minimum_observed_cell_n": int(labels.value_counts().min()) if labels.notna().any() else 0,
        "reason": reason,
    })
    return labels, method, reason


def strata_summary(frame, feature, analysis_name, population, context=None):
    labels, method, reason = adaptive_strata(frame, feature, analysis_name, population)
    if context:
        stratification_qc_rows[-1].update(context)
    rows = []
    if method == "unavailable":
        row = {
            "analysis_name": analysis_name, "population": population, "feature": feature,
            "stratification_method": method, "stratum": "unavailable", "availability_reason": reason,
            "n_cases": 0, "n_users": 0, "mean_delta_ndcg_at_5": np.nan,
            "median_delta_ndcg_at_5": np.nan, "bootstrap_ci_95_low": np.nan,
            "bootstrap_ci_95_high": np.nan,
        }
        if context:
            row.update(context)
        return [row]
    work = frame.assign(_stratum=labels)
    for stratum, subset in work.dropna(subset=["_stratum"]).groupby("_stratum", sort=True, observed=True):
        row = {
            "analysis_name": analysis_name, "population": population, "feature": feature,
            "stratification_method": method, "stratum": str(stratum), "availability_reason": reason,
            **continuous_summary(subset, "delta_ndcg_at_5", f"{analysis_name}|{population}|{feature}|{stratum}"),
        }
        if context:
            row.update(context)
        rows.append(row)
    return rows


def population_subsets(frame):
    return {
        "overall": frame,
        "Strong": frame.loc[frame["regime"].astype(str).str.lower().eq("strong")],
    }


In [46]:
# ==== Continuous Stage deltas, alignment, interaction, and QCHS Status ====
continuous_summary_rows = []
alignment_rows = []
stage1_qchs_status_rows = []
interaction_rows = []

for (contrast, family), contrast_frame in stage_master.groupby(["contrast_name", "reranker_family"], sort=False, observed=True):
    for population, subset in population_subsets(contrast_frame).items():
        continuous_summary_rows.append({
            "category_id": CATEGORY_ID, "contrast_name": contrast,
            "effect_label": STAGE_EFFECT_LABELS[contrast], "reranker_family": family,
            "reranker_role": "primary" if family == PRIMARY_RERANKER else ("secondary" if family == SECONDARY_RERANKER else "shared_stage1"),
            "population": population, "pool_depth": BATCH_A_POOL_DEPTH, "metric": PRIMARY_METRIC,
            **continuous_summary(subset, "delta_ndcg_at_5", f"stage|{contrast}|{family}|{population}"),
        })
        current_alignment_rows = strata_summary(
            subset, coherence_feature, "alignment_heterogeneity", population,
            context={
                "category_id": CATEGORY_ID, "contrast_name": contrast,
                "reranker_family": family, "pool_depth": BATCH_A_POOL_DEPTH,
            },
        )
        alignment_rows.extend(current_alignment_rows)

        status_work = subset.copy()
        if len(status_work):
            for status, status_subset in status_work.groupby("stage1_qchs_status", observed=True):
                stage1_qchs_status_rows.append({
                    "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                    "population": population, "stage1_qchs_status": status, "comparison_row_type": "within_status",
                    **continuous_summary(status_subset, "delta_ndcg_at_5", f"qchs_status|{contrast}|{family}|{population}|{status}"),
                })
            gap, low, high = cluster_bootstrap_group_gap(
                status_work, "delta_ndcg_at_5", "stage1_qchs_status", "Stage1-QCHS-active", "Stage1-QCHS-fallback",
                f"qchs_gap|{contrast}|{family}|{population}",
            )
            stage1_qchs_status_rows.append({
                "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                "population": population, "stage1_qchs_status": "QCHS-active_minus_fallback",
                "comparison_row_type": "descriptive_group_gap", "n_cases": len(status_work),
                "n_users": status_work["user_id"].nunique(), "mean_delta_ndcg_at_5": gap,
                "median_delta_ndcg_at_5": np.nan, "bootstrap_ci_95_low": low,
                "bootstrap_ci_95_high": high, "positive_delta_rate_descriptive": np.nan,
                "zero_delta_rate_descriptive": np.nan, "negative_delta_rate_descriptive": np.nan,
                "bootstrap_unit": "user_id cluster", "bootstrap_repetitions": BOOTSTRAP_REPS,
            })

        interaction_work = subset[["case_id", "user_id", "delta_ndcg_at_5", history_quantity_feature, coherence_feature]].copy()
        interaction_work = interaction_work.dropna(subset=[history_quantity_feature, coherence_feature, "delta_ndcg_at_5"])
        if len(interaction_work):
            history_median = float(pd.to_numeric(interaction_work[history_quantity_feature], errors="coerce").median())
            coherence_median = float(pd.to_numeric(interaction_work[coherence_feature], errors="coerce").median())
            interaction_work["history_level"] = np.where(
                pd.to_numeric(interaction_work[history_quantity_feature], errors="coerce").le(history_median), "low", "high"
            )
            interaction_work["coherence_level"] = np.where(
                pd.to_numeric(interaction_work[coherence_feature], errors="coerce").le(coherence_median), "low", "high"
            )
            interaction_work["interaction_cell"] = interaction_work["history_level"] + "_history__" + interaction_work["coherence_level"] + "_coherence"
            cell_counts = interaction_work["interaction_cell"].value_counts()
            expected_cells = {"low_history__low_coherence", "low_history__high_coherence", "high_history__low_coherence", "high_history__high_coherence"}
            available = set(cell_counts.index) == expected_cells and int(cell_counts.min()) >= MIN_INTERACTION_CELL_N
            stratification_qc_rows.append({
                "analysis_name": "history_quantity_x_coherence", "population": population,
                "feature": f"{history_quantity_feature} × {coherence_feature}",
                "available_case_count": len(interaction_work), "unique_value_count": interaction_work["interaction_cell"].nunique(),
                "method": "median_2x2" if available else "unavailable", "minimum_cell_n": MIN_INTERACTION_CELL_N,
                "minimum_observed_cell_n": int(cell_counts.min()) if len(cell_counts) else 0,
                "reason": "descriptive 2×2 interaction" if available else "one or more 2×2 cells are too small",
            })
            if available:
                interaction_estimate, interaction_low, interaction_high = cluster_bootstrap_four_cell_interaction(
                    interaction_work, "delta_ndcg_at_5", "interaction_cell",
                    f"interaction_gap|{contrast}|{family}|{population}",
                )
                for cell, cell_subset in interaction_work.groupby("interaction_cell", observed=True):
                    interaction_rows.append({
                        "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                        "population": population, "row_type": "cell", "interaction_cell": cell,
                        "history_split_value": history_median, "coherence_split_value": coherence_median,
                        **continuous_summary(cell_subset, "delta_ndcg_at_5", f"interaction|{contrast}|{family}|{population}|{cell}"),
                    })
                interaction_rows.append({
                    "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                    "population": population, "row_type": "difference_in_differences_descriptive",
                    "interaction_cell": "HH-HL-LH+LL", "history_split_value": history_median,
                    "coherence_split_value": coherence_median, "n_cases": len(interaction_work),
                    "n_users": interaction_work["user_id"].nunique(),
                    "mean_delta_ndcg_at_5": interaction_estimate, "median_delta_ndcg_at_5": np.nan,
                    "bootstrap_ci_95_low": interaction_low, "bootstrap_ci_95_high": interaction_high,
                    "positive_delta_rate_descriptive": np.nan, "zero_delta_rate_descriptive": np.nan,
                    "negative_delta_rate_descriptive": np.nan, "bootstrap_unit": "user_id cluster",
                    "bootstrap_repetitions": BOOTSTRAP_REPS,
                })
            else:
                interaction_rows.append({
                    "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                    "population": population, "row_type": "unavailable", "interaction_cell": "",
                    "history_split_value": history_median, "coherence_split_value": coherence_median,
                    "n_cases": len(interaction_work), "n_users": interaction_work["user_id"].nunique(),
                    "mean_delta_ndcg_at_5": np.nan, "median_delta_ndcg_at_5": np.nan,
                    "bootstrap_ci_95_low": np.nan, "bootstrap_ci_95_high": np.nan,
                    "positive_delta_rate_descriptive": np.nan, "zero_delta_rate_descriptive": np.nan,
                    "negative_delta_rate_descriptive": np.nan, "bootstrap_unit": "not run: insufficient cell size",
                    "bootstrap_repetitions": 0,
                })

continuous_delta_summary = pd.DataFrame(continuous_summary_rows)
alignment_strata_summary = pd.DataFrame(alignment_rows)
history_coherence_interaction = pd.DataFrame(interaction_rows)
stage1_qchs_active_fallback_summary = pd.DataFrame(stage1_qchs_status_rows)

display(continuous_delta_summary)
display(alignment_strata_summary.head(12))


,category_id,contrast_name,effect_label,reranker_family,reranker_role,population,pool_depth,metric,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions
0,face,P1-only_minus_P0,Stage 1 personalization effect,shared_stage1,shared_stage1,overall,1000,NDCG@5,2288,2288,0.001613,0.0,-0.000409,0.003646,0.006556,0.989510,0.003934,user_id cluster over paired case-level deltas,2000
1,face,P1-only_minus_P0,Stage 1 personalization effect,shared_stage1,shared_stage1,Strong,1000,NDCG@5,572,572,0.003046,0.0,-0.002067,0.009013,0.012238,0.979021,0.008741,user_id cluster over paired case-level deltas,2000
2,face,P2-P_minus_P2-Q,Stage 2 prior-feature effect,lightgbm,primary,overall,1000,NDCG@5,2288,2288,0.011492,0.0,0.006689,0.016462,0.036276,0.949301,0.014423,user_id cluster over paired case-level deltas,2000
3,face,P2-P_minus_P2-Q,Stage 2 prior-feature effect,lightgbm,primary,Strong,1000,NDCG@5,572,572,0.009965,0.0,0.000262,0.019883,0.045455,0.933566,0.020979,user_id cluster over paired case-level deltas,2000
4,face,Full_minus_P2-P,Personalized candidate-source effect,lightgbm,primary,overall,1000,NDCG@5,2288,2288,-0.004068,0.0,-0.009567,0.001205,0.024913,0.940997,0.034091,user_id cluster over paired case-level deltas,2000
5,face,Full_minus_P2-P,Personalized candidate-source effect,lightgbm,primary,Strong,1000,NDCG@5,572,572,-0.000882,0.0,-0.012042,0.010030,0.034965,0.926573,0.038462,user_id cluster over paired case-level deltas,2000
6,face,Full_minus_P2-Q,Combined personalization effect,lightgbm,primary,overall,1000,NDCG@5,2288,2288,0.007423,0.0,0.001999,0.012972,0.035839,0.941871,0.022290,user_id cluster over paired case-level deltas,2000
7,face,Full_minus_P2-Q,Combined personalization effect,lightgbm,primary,Strong,1000,NDCG@5,572,572,0.009083,0.0,-0.003069,0.020860,0.047203,0.917832,0.034965,user_id cluster over paired case-level deltas,2000
8,face,P2-P_minus_P2-Q,Stage 2 prior-feature effect,transformer,secondary,overall,1000,NDCG@5,2288,2288,0.005444,0.0,-0.000477,0.011761,0.031469,0.946241,0.022290,user_id cluster over paired case-level deltas,2000
9,face,P2-P_minus_P2-Q,Stage 2 prior-feature effect,transformer,secondary,Strong,1000,NDCG@5,572,572,-0.011447,0.0,-0.026424,0.003349,0.033217,0.921329,0.045455,user_id cluster over paired case-level deltas,2000


,analysis_name,population,feature,stratification_method,stratum,availability_reason,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions,category_id,contrast_name,reranker_family,pool_depth
0,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q1,four qcut cells satisfy the minimum size,429,429,0.002697,0.0,-0.002835,0.008619,0.013986,0.979021,0.006993,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
1,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q2,four qcut cells satisfy the minimum size,430,430,0.005939,0.0,0.000448,0.012220,0.013953,0.983721,0.002326,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
2,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q3,four qcut cells satisfy the minimum size,428,428,-0.000616,0.0,-0.007874,0.005491,0.004673,0.985981,0.009346,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
3,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q4,four qcut cells satisfy the minimum size,429,429,0.000569,0.0,-0.002581,0.004288,0.002331,0.995338,0.002331,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
4,alignment_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q1,four qcut cells satisfy the minimum size,143,143,0.004594,0.0,-0.008116,0.019525,0.020979,0.965035,0.013986,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
5,alignment_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q2,four qcut cells satisfy the minimum size,144,144,0.004809,0.0,-0.005126,0.017735,0.013889,0.979167,0.006944,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
6,alignment_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q3,four qcut cells satisfy the minimum size,143,143,0.005328,0.0,-0.005162,0.018398,0.013986,0.979021,0.006993,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
7,alignment_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q4,four qcut cells satisfy the minimum size,142,142,-0.002599,0.0,-0.007797,0.000000,0.000000,0.992958,0.007042,user_id cluster over paired case-level deltas,2000,face,P1-only_minus_P0,shared_stage1,1000
8,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q1,four qcut cells satisfy the minimum size,429,429,0.020558,0.0,0.007158,0.035519,0.055944,0.930070,0.013986,user_id cluster over paired case-level deltas,2000,face,P2-P_minus_P2-Q,lightgbm,1000
9,alignment_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q2,four qcut cells satisfy the minimum size,430,430,0.009141,0.0,0.000752,0.018564,0.034884,0.946512,0.018605,user_id cluster over paired case-level deltas,2000,face,P2-P_minus_P2-Q,lightgbm,1000


In [47]:
# ==== QCHS Versus All Prior, headroom/item evidence, and Brand Heterogeneity ====
prior_policy_rows = []
for diagnostic_frame in [d1_qchs_all, d2_diagnostic.loc[d2_diagnostic["contrast_name"].eq("QCHS_Filtered_minus_Actual_All_Prior")]]:
    if diagnostic_frame.empty:
        continue
    for (contrast, family), contrast_frame in diagnostic_frame.groupby(["contrast_name", "reranker_family"], observed=True):
        for population, subset in population_subsets(contrast_frame).items():
            for feature in [coherence_feature, history_quantity_feature]:
                rows = strata_summary(
                    subset, feature, "qchs_vs_all_prior_heterogeneity", population,
                    context={
                        "category_id": CATEGORY_ID, "contrast_name": contrast,
                        "reranker_family": family, "pool_depth": PRIMARY_POOL_DEPTH,
                    },
                )
                for row in rows:
                    row.update({
                        "category_id": CATEGORY_ID, "contrast_name": contrast,
                        "reranker_family": family, "pool_depth": PRIMARY_POOL_DEPTH,
                        "interpretation": "descriptive heterogeneity; not a causal mechanism estimate",
                    })
                prior_policy_rows.extend(rows)
qchs_all_prior_heterogeneity = pd.DataFrame(prior_policy_rows)

headroom_rows = []
for (contrast, family), contrast_frame in stage_master.groupby(["contrast_name", "reranker_family"], observed=True):
    for population, subset in population_subsets(contrast_frame).items():
        rows = strata_summary(
            subset, "baseline_headroom_ndcg_at_5", "baseline_headroom", population,
            context={
                "category_id": CATEGORY_ID, "contrast_name": contrast,
                "reranker_family": family, "pool_depth": BATCH_A_POOL_DEPTH,
            },
        )
        for row in rows:
            row.update({"category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family, "row_type": "stratum"})
        headroom_rows.extend(rows)
        for feature in ["baseline_headroom_ndcg_at_5", *item_evidence_features]:
            correlation, n_pairs = spearman_descriptive(subset, feature)
            headroom_rows.append({
                "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                "analysis_name": "baseline_item_evidence_correlation", "population": population,
                "feature": feature, "stratification_method": "none", "stratum": "continuous",
                "availability_reason": "descriptive Spearman rank correlation",
                "row_type": "correlation", "n_cases": n_pairs, "n_users": subset["user_id"].nunique(),
                "mean_delta_ndcg_at_5": np.nan, "median_delta_ndcg_at_5": np.nan,
                "bootstrap_ci_95_low": np.nan, "bootstrap_ci_95_high": np.nan,
                "spearman_rho": correlation,
            })
        for feature in item_evidence_features:
            rows = strata_summary(
                subset, feature, "baseline_item_evidence_strata", population,
                context={
                    "category_id": CATEGORY_ID, "contrast_name": contrast,
                    "reranker_family": family, "pool_depth": BATCH_A_POOL_DEPTH,
                },
            )
            for row in rows:
                row.update({"category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family, "row_type": "stratum"})
            headroom_rows.extend(rows)
baseline_item_evidence_headroom = pd.DataFrame(headroom_rows)

brand_rows = []
brand_analysis_frames = [stage_master]
brand_ablation = d2_diagnostic.loc[d2_diagnostic["contrast_name"].eq("Actual_All_Prior_minus_No_User_Brand")].copy()
if not brand_ablation.empty:
    brand_analysis_frames.append(brand_ablation)
for analysis_frame in brand_analysis_frames:
    # Depth follows the provenance of the frame, not a global constant:
    # the Batch D2 brand ablation was executed at PRIMARY_POOL_DEPTH while
    # the Batch A stage master is reported at BATCH_A_POOL_DEPTH.
    frame_pool_depth = (
        PRIMARY_POOL_DEPTH
        if (
            "analysis_source" in analysis_frame.columns
            and analysis_frame["analysis_source"].astype(str).str.startswith("Batch D").any()
        )
        else BATCH_A_POOL_DEPTH
    )
    for (contrast, family), contrast_frame in analysis_frame.groupby(["contrast_name", "reranker_family"], observed=True):
        for population, subset in population_subsets(contrast_frame).items():
            if not brand_features:
                brand_rows.append({
                    "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                    "analysis_name": "brand_coherence_heterogeneity", "population": population,
                    "feature": "", "stratification_method": "unavailable", "stratum": "unavailable",
                    "availability_reason": "No standardized Batch B brand-coherence feature resolved",
                    "n_cases": 0, "n_users": 0, "mean_delta_ndcg_at_5": np.nan,
                    "median_delta_ndcg_at_5": np.nan, "bootstrap_ci_95_low": np.nan,
                    "bootstrap_ci_95_high": np.nan,
                })
            for feature in brand_features:
                rows = strata_summary(
                    subset, feature, "brand_coherence_heterogeneity", population,
                    context={
                        "category_id": CATEGORY_ID, "contrast_name": contrast,
                        "reranker_family": family, "pool_depth": frame_pool_depth,
                    },
                )
                for row in rows:
                    row.update({
                        "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                        "interpretation": "descriptive association; item-side brand representation remains retained",
                    })
                brand_rows.extend(rows)
brand_coherence_heterogeneity = pd.DataFrame(brand_rows)

display(qchs_all_prior_heterogeneity.head(12))
display(baseline_item_evidence_headroom.head(12))
display(brand_coherence_heterogeneity.head(12))


,analysis_name,population,feature,stratification_method,stratum,availability_reason,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions,category_id,contrast_name,reranker_family,pool_depth,interpretation
0,qchs_vs_all_prior_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q1,four qcut cells satisfy the minimum size,429,429,0.002128,0.0,-0.000323,0.005847,0.004662,0.993007,0.002331,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
1,qchs_vs_all_prior_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q2,four qcut cells satisfy the minimum size,430,430,0.003036,0.0,-0.001209,0.009436,0.009302,0.986047,0.004651,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
2,qchs_vs_all_prior_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q3,four qcut cells satisfy the minimum size,428,428,-0.002031,0.0,-0.005697,0.000936,0.002336,0.990654,0.007009,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
3,qchs_vs_all_prior_heterogeneity,overall,b_preference_structurality_features_by_case__q...,quartiles,Q4,four qcut cells satisfy the minimum size,429,429,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
4,qchs_vs_all_prior_heterogeneity,overall,b_case_alignment_features__strict_prior_intera...,quartiles,Q1,four qcut cells satisfy the minimum size,572,572,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
5,qchs_vs_all_prior_heterogeneity,overall,b_case_alignment_features__strict_prior_intera...,quartiles,Q2,four qcut cells satisfy the minimum size,572,572,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
6,qchs_vs_all_prior_heterogeneity,overall,b_case_alignment_features__strict_prior_intera...,quartiles,Q3,four qcut cells satisfy the minimum size,572,572,-0.000592,0.0,-0.003107,0.001451,0.001748,0.991259,0.006993,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
7,qchs_vs_all_prior_heterogeneity,overall,b_case_alignment_features__strict_prior_intera...,quartiles,Q4,four qcut cells satisfy the minimum size,572,572,0.002952,0.0,-0.001496,0.008319,0.010490,0.986014,0.003497,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
8,qchs_vs_all_prior_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q1,four qcut cells satisfy the minimum size,143,143,0.004288,0.0,0.000000,0.012863,0.006993,0.993007,0.000000,user_id cluster over paired case-level deltas,2000,face,QCHS_minus_All_Prior,stage1_controlled_retrieval,1000,descriptive heterogeneity; not a causal mechan...
9,qchs_vs_all_prior_heterogeneity,Strong,b_preference_structurality_features_by_case__q...,quartiles,Q2,four qcut cells satisfy the minimum size,144,144,0.001818,0.0,-0.006298,0.010363,0.013889,0.979167,0.006944,user_id cluster over paired case-level deltas,2000,f

,analysis_name,population,feature,stratification_method,stratum,availability_reason,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,category_id,contrast_name,reranker_family,pool_depth,row_type,spearman_rho,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions
0,baseline_headroom,overall,baseline_headroom_ndcg_at_5,unavailable,unavailable,quartiles and median high/low both fail minimu...,0,0,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,1000.0,stratum,NaN,NaN,NaN,NaN,NaN,NaN
1,baseline_item_evidence_correlation,overall,baseline_headroom_ndcg_at_5,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,0.413191,NaN,NaN,NaN,NaN,NaN
2,baseline_item_evidence_correlation,overall,b_preference_structurality_features_by_case__q...,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,-0.022711,NaN,NaN,NaN,NaN,NaN
3,baseline_item_evidence_correlation,overall,b_preference_structurality_features_by_case__q...,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,-0.033226,NaN,NaN,NaN,NaN,NaN
4,baseline_item_evidence_correlation,overall,b_case_alignment_features__query_source_signal...,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,-0.024724,NaN,NaN,NaN,NaN,NaN
5,baseline_item_evidence_correlation,overall,b_preference_structurality_features_by_case__c...,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,0.022413,NaN,NaN,NaN,NaN,NaN
6,baseline_item_evidence_correlation,overall,b_preference_structurality_features_by_case__r...,none,continuous,descriptive Spearman rank correlation,2288,2288,NaN,NaN,NaN,NaN,face,Full_minus_P2-P,lightgbm,NaN,correlation,0.030717,NaN,NaN,NaN,NaN,NaN
7,baseline_item_evidence_strata,overall,b_preference_structurality_features_by_case__q...,median_high_low,high,quartile cells were insufficient; split at med...,1075,1075,-0.004266,0.0,-0.012790,0.004756,face,Full_minus_P2-P,lightgbm,1000.0,stratum,NaN,0.031628,0.926512,0.041860,user_id cluster over paired case-level deltas,2000.0
8,baseline_item_evidence_strata,overall,b_preference_structurality_features_by_case__q...,median_high_low,low,quartile cells were insufficient; split at med...,1213,1213,-0.003893,0.0,-0.010380,0.002875,face,Full_minus_P2-P,lightgbm,1000.0,stratum,NaN,0.018961,0.953833,0.027205,user_id cluster over paired case-level deltas,2000.0
9,baseline_item_evidence_strata,overall,b_preference_structurality_features_by_case__q...,median_high_low,high,quartile cells were insufficient; split at med...,214,214,-0.026573,0.0,-0.052648,-0.000860,face,Full_minus_P2-P,lightgbm,1000.0,stratum,NaN,0.023364,0.897196,0.079439,user_id cluster over paired case-level deltas,2000.0


,analysis_name,population,feature,stratification_method,stratum,availability_reason,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions,category_id,contrast_name,reranker_family,pool_depth,interpretation
0,brand_coherence_heterogeneity,overall,b_case_alignment_features__dominant_brand_share,quartiles,Q1,four qcut cells satisfy the minimum size,443,443,-0.004202,0.0,-0.016758,0.007414,0.029345,0.932280,0.038375,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
1,brand_coherence_heterogeneity,overall,b_case_alignment_features__dominant_brand_share,quartiles,Q2,four qcut cells satisfy the minimum size,436,436,0.005273,0.0,-0.008985,0.018643,0.045872,0.919725,0.034404,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
2,brand_coherence_heterogeneity,overall,b_case_alignment_features__dominant_brand_share,quartiles,Q3,four qcut cells satisfy the minimum size,460,460,-0.019311,0.0,-0.031997,-0.007948,0.013043,0.939130,0.047826,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
3,brand_coherence_heterogeneity,overall,b_case_alignment_features__dominant_brand_share,quartiles,Q4,four qcut cells satisfy the minimum size,370,370,-0.002333,0.0,-0.022457,0.017391,0.048649,0.886486,0.064865,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
4,brand_coherence_heterogeneity,overall,b_preference_structurality_features_by_case__b...,quartiles,Q1,four qcut cells satisfy the minimum size,430,430,-0.005598,0.0,-0.024650,0.013206,0.041860,0.895349,0.062791,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
5,brand_coherence_heterogeneity,overall,b_preference_structurality_features_by_case__b...,quartiles,Q2,four qcut cells satisfy the minimum size,428,428,-0.003102,0.0,-0.015780,0.010438,0.032710,0.927570,0.039720,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
6,brand_coherence_heterogeneity,overall,b_preference_structurality_features_by_case__b...,quartiles,Q3,four qcut cells satisfy the minimum size,688,688,-0.006585,0.0,-0.016557,0.002987,0.027616,0.934593,0.037791,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
7,brand_coherence_heterogeneity,overall,b_preference_structurality_features_by_case__b...,quartiles,Q4,four qcut cells satisfy the minimum size,163,163,-0.006404,0.0,-0.032200,0.019503,0.036810,0.914110,0.049080,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
8,brand_coherence_heterogeneity,overall,b_case_alignment_features__unique_prior_brand_...,median_high_low,high,quartile cells were insufficient; split at med...,1133,1133,-0.005450,0.0,-0.013586,0.002742,0.031774,0.928508,0.039718,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...
9,brand_coherence_heterogeneity,overall,b_case_alignment_features__unique_prior_brand_...,median_high_low,low,quartile cells were insufficient; split at med...,1155,1155,-0.002713,0.0,-0.009920,0.004469,0.018182,0.953247,0.028571,user_id cluster over paired case-level deltas,2000,face,Full_minus_P2-P,lightgbm,1000,descriptive association; item-side brand repre...


In [48]:
# ==== Continuous-delta Error Taxonomy and strong-history Support Classification ====
error_taxonomy = stage_master.copy()
error_taxonomy["direction_descriptive"] = np.select(
    [error_taxonomy["delta_ndcg_at_5"].gt(0.0), error_taxonomy["delta_ndcg_at_5"].lt(0.0)],
    ["gain", "loss"], default="zero",
)
error_taxonomy["magnitude_band_descriptive"] = "zero"
for (contrast, family), index in error_taxonomy.groupby(["contrast_name", "reranker_family"], observed=True).groups.items():
    absolute_nonzero = error_taxonomy.loc[index, "delta_ndcg_at_5"].abs()
    absolute_nonzero = absolute_nonzero.loc[absolute_nonzero.gt(0.0)]
    if absolute_nonzero.empty:
        continue
    threshold = float(absolute_nonzero.median())
    delta = error_taxonomy.loc[index, "delta_ndcg_at_5"]
    error_taxonomy.loc[index, "magnitude_band_descriptive"] = np.select(
        [
            delta.lt(0.0) & delta.abs().gt(threshold),
            delta.lt(0.0),
            delta.gt(0.0) & delta.abs().gt(threshold),
            delta.gt(0.0),
        ],
        ["larger_loss", "smaller_loss", "larger_gain", "smaller_gain"],
        default="zero",
    )
baseline_rank_left = pd.to_numeric(
    error_taxonomy["target_rank_left"], errors="coerce"
)
baseline_in_top5 = (
    baseline_rank_left.ge(1)
    & baseline_rank_left.le(PRIMARY_K)
).fillna(False).astype(bool)
error_taxonomy["baseline_top5_status"] = np.where(
    baseline_in_top5,
    "baseline_target_in_top5", "baseline_target_outside_top5",
)
error_taxonomy["primary_analysis_value"] = error_taxonomy["delta_ndcg_at_5"]
error_taxonomy["won_lost_status"] = "descriptive_only"

taxonomy_summary_rows = []
for (contrast, family), contrast_frame in error_taxonomy.groupby(["contrast_name", "reranker_family"], observed=True):
    for population, subset in population_subsets(contrast_frame).items():
        for (direction, magnitude, baseline_status), cell in subset.groupby(
            ["direction_descriptive", "magnitude_band_descriptive", "baseline_top5_status"], observed=True
        ):
            taxonomy_summary_rows.append({
                "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                "population": population, "direction_descriptive": direction,
                "magnitude_band_descriptive": magnitude, "baseline_top5_status": baseline_status,
                "won_lost_status": "descriptive_only",
                **continuous_summary(cell, "delta_ndcg_at_5", f"taxonomy|{contrast}|{family}|{population}|{direction}|{magnitude}|{baseline_status}"),
            })
continuous_delta_error_taxonomy_summary = pd.DataFrame(taxonomy_summary_rows)

strong_rows = []
mechanism_gap_features = [
    "baseline_headroom_ndcg_at_5", history_quantity_feature, coherence_feature, "stage1_qchs_active", *brand_features[:2]
]
diagnostic_for_strong_history = pd.concat([
    stage_master,
    d1_qchs_all,
    d2_diagnostic.loc[d2_diagnostic["contrast_name"].eq("QCHS_Filtered_minus_Actual_All_Prior")],
], ignore_index=True, sort=False)
for (contrast, family), contrast_frame in diagnostic_for_strong_history.groupby(["contrast_name", "reranker_family"], observed=True):
    overall = contrast_frame
    strong = contrast_frame.loc[contrast_frame["regime"].astype(str).str.lower().eq("strong")]
    non_strong = contrast_frame.loc[~contrast_frame["regime"].astype(str).str.lower().eq("strong")]
    overall_stats = continuous_summary(overall, "delta_ndcg_at_5", f"strong_history|{contrast}|{family}|overall")
    strong_stats = continuous_summary(strong, "delta_ndcg_at_5", f"strong_history|{contrast}|{family}|strong")
    non_strong_stats = continuous_summary(non_strong, "delta_ndcg_at_5", f"strong_history|{contrast}|{family}|non_strong")
    comparison = contrast_frame.assign(strong_group=np.where(
        contrast_frame["regime"].astype(str).str.lower().eq("strong"), "Strong", "non-Strong"
    ))
    gap, gap_low, gap_high = cluster_bootstrap_group_gap(
        comparison, "delta_ndcg_at_5", "strong_group", "Strong", "non-Strong",
        f"strong_gap|{contrast}|{family}",
    )
    row = {
        "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
        "pool_depth": BATCH_A_POOL_DEPTH, "metric": PRIMARY_METRIC,
        "overall_n_cases": overall_stats["n_cases"], "overall_mean_delta": overall_stats["mean_delta_ndcg_at_5"],
        "overall_ci_low": overall_stats["bootstrap_ci_95_low"], "overall_ci_high": overall_stats["bootstrap_ci_95_high"],
        "strong_n_cases": strong_stats["n_cases"], "strong_mean_delta": strong_stats["mean_delta_ndcg_at_5"],
        "strong_ci_low": strong_stats["bootstrap_ci_95_low"], "strong_ci_high": strong_stats["bootstrap_ci_95_high"],
        "non_strong_n_cases": non_strong_stats["n_cases"], "non_strong_mean_delta": non_strong_stats["mean_delta_ndcg_at_5"],
        "strong_minus_non_strong_mean_delta": gap,
        "strong_minus_non_strong_ci_low": gap_low, "strong_minus_non_strong_ci_high": gap_high,
        "strong_history_support_classification": (
            "supported strong below weak"
            if pd.notna(gap_low) and pd.notna(gap_high) and gap_high < 0
            else "supported reversal"
            if pd.notna(gap_low) and pd.notna(gap_high) and gap_low > 0
            else "descriptive strong below weak"
            if pd.notna(gap) and gap < 0
            else "no supported strong-history premium"
        ),
        "strong_history_claim_limited_to_ci_support": True,
        "causal_explanation_claimed": False,
    }
    for feature in mechanism_gap_features:
        if feature not in contrast_frame.columns:
            continue
        numeric = pd.to_numeric(contrast_frame[feature], errors="coerce")
        strong_mean = float(numeric.loc[contrast_frame["regime"].astype(str).str.lower().eq("strong")].mean())
        non_strong_mean = float(numeric.loc[~contrast_frame["regime"].astype(str).str.lower().eq("strong")].mean())
        safe_name = normalized_name(feature)[:70]
        row[f"strong_mean__{safe_name}"] = strong_mean
        row[f"non_strong_mean__{safe_name}"] = non_strong_mean
        row[f"strong_minus_non_strong__{safe_name}"] = strong_mean - non_strong_mean
    strong_rows.append(row)
strong_history_premium_summary = pd.DataFrame(strong_rows)

display(continuous_delta_error_taxonomy_summary.head(16))
display(strong_history_premium_summary)


,category_id,contrast_name,reranker_family,population,direction_descriptive,magnitude_band_descriptive,baseline_top5_status,won_lost_status,n_cases,n_users,mean_delta_ndcg_at_5,median_delta_ndcg_at_5,bootstrap_ci_95_low,bootstrap_ci_95_high,positive_delta_rate_descriptive,zero_delta_rate_descriptive,negative_delta_rate_descriptive,bootstrap_unit,bootstrap_repetitions
0,face,Full_minus_P2-P,lightgbm,overall,gain,larger_gain,baseline_target_in_top5,descriptive_only,3,3,0.560824,0.569323,0.500000,0.613147,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000
1,face,Full_minus_P2-P,lightgbm,overall,gain,larger_gain,baseline_target_outside_top5,descriptive_only,24,24,0.705843,0.630930,0.627965,0.783721,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000
2,face,Full_minus_P2-P,lightgbm,overall,gain,smaller_gain,baseline_target_in_top5,descriptive_only,15,15,0.263074,0.369070,0.203125,0.321442,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000
3,face,Full_minus_P2-P,lightgbm,overall,gain,smaller_gain,baseline_target_outside_top5,descriptive_only,15,15,0.404382,0.386853,0.395618,0.416069,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000
4,face,Full_minus_P2-P,lightgbm,overall,loss,larger_loss,baseline_target_in_top5,descriptive_only,38,38,-0.665458,-0.622038,-0.727859,-0.608077,0.0,0.0,1.0,user_id cluster over paired case-level deltas,2000
5,face,Full_minus_P2-P,lightgbm,overall,loss,smaller_loss,baseline_target_in_top5,descriptive_only,40,40,-0.316392,-0.369070,-0.354323,-0.275899,0.0,0.0,1.0,user_id cluster over paired case-level deltas,2000
6,face,Full_minus_P2-P,lightgbm,overall,zero,zero,baseline_target_in_top5,descriptive_only,68,68,0.000000,0.000000,0.000000,0.000000,0.0,1.0,0.0,user_id cluster over paired case-level deltas,2000
7,face,Full_minus_P2-P,lightgbm,overall,zero,zero,baseline_target_outside_top5,descriptive_only,2085,2085,0.000000,0.000000,0.000000,0.000000,0.0,1.0,0.0,user_id cluster over paired case-level deltas,2000
8,face,Full_minus_P2-P,lightgbm,Strong,gain,larger_gain,baseline_target_in_top5,descriptive_only,1,1,0.500000,0.500000,0.500000,0.500000,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000
9,face,Full_minus_P2-P,lightgbm,Strong,gain,larger_gain,baseline_target_outside_top5,descriptive_only,9,9,0.669302,0.630930,0.558191,0.794961,1.0,0.0,0.0,user_id cluster over paired case-level deltas,2000


,category_id,contrast_name,reranker_family,pool_depth,metric,overall_n_cases,overall_mean_delta,overall_ci_low,overall_ci_high,strong_n_cases,strong_mean_delta,strong_ci_low,strong_ci_high,non_strong_n_cases,non_strong_mean_delta,strong_minus_non_strong_mean_delta,strong_minus_non_strong_ci_low,strong_minus_non_strong_ci_high,strong_history_support_classification,strong_history_claim_limited_to_ci_support,causal_explanation_claimed,strong_mean__baseline_headroom_ndcg_at_5,non_strong_mean__baseline_headroom_ndcg_at_5,strong_minus_non_strong__baseline_headroom_ndcg_at_5,strong_mean__b_case_alignment_features_strict_prior_interaction_count,non_strong_mean__b_case_alignment_features_strict_prior_interaction_count,strong_minus_non_strong__b_case_alignment_features_strict_prior_interaction_count,strong_mean__b_preference_structurality_features_by_case_query_profile_functional_a,non_strong_mean__b_preference_structurality_features_by_case_query_profile_functional_a,strong_minus_non_strong__b_preference_structurality_features_by_case_query_profile_functional_a,strong_mean__stage1_qchs_active,non_strong_mean__stage1_qchs_active,strong_minus_non_strong__stage1_qchs_active,strong_mean__b_case_alignment_features_dominant_brand_share,non_strong_mean__b_case_alignment_features_dominant_brand_share,strong_minus_non_strong__b_case_alignment_features_dominant_brand_share,strong_mean__b_preference_structurality_features_by_case_brand_entropy,non_strong_mean__b_preference_structurality_features_by_case_brand_entropy,strong_minus_non_strong__b_preference_structurality_features_by_case_brand_entropy
0,face,Full_minus_P2-P,lightgbm,1000,NDCG@5,2288,-0.004068,-0.009514,0.001208,572,-0.000882,-0.012001,0.010548,1716,-0.005130,0.004248,-0.008489,0.017431,no supported strong-history premium,True,False,0.926631,0.953599,-0.026968,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
1,face,Full_minus_P2-P,transformer,1000,NDCG@5,2288,0.002060,-0.003433,0.007461,572,0.001832,-0.008953,0.013529,1716,0.002136,-0.000304,-0.013614,0.013359,descriptive strong below weak,True,False,0.949026,0.967143,-0.018116,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
2,face,Full_minus_P2-Q,lightgbm,1000,NDCG@5,2288,0.007423,0.001724,0.012931,572,0.009083,-0.002707,0.021881,1716,0.006870,0.002213,-0.011280,0.016343,no supported strong-history premium,True,False,0.936597,0.965600,-0.029003,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
3,face,Full_minus_P2-Q,transformer,1000,NDCG@5,2288,0.007504,0.000751,0.014372,572,-0.009614,-0.024420,0.004700,1716,0.013210,-0.022825,-0.039421,-0.006100,supported strong below weak,True,False,0.937580,0.978217,-0.040637,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
4,face,P1-only_minus_P0,shared_stage1,1000,NDCG@5,2288,0.001613,-0.000440,0.003658,572,0.003046,-0.002140,0.008936,1716,0.001136,0.001910,-0.003374,0.007811,no supported strong-history premium,True,False,0.979538,0.995530,-0.015992,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
5,face,P2-P_minus_P2-Q,lightgbm,1000,NDCG@5,2288,0.011492,0.006656,0.016397,572,0.009965,0.000481,0.019638,1716,0.012001,-0.002035,-0.013158,0.009063,descriptive strong below weak,True,False,0.936597,0.965600,-0.029003,42.839161,2.751166,40.087995,0.026133,0.042464,-0.01633,0.979021,0.555361,0.42366,0.1041,0.502728,-0.398628,0.980317,0.701786,0.278531
6,face,P2-P_minus_P2-Q,transformer,1000,NDCG@5,2288,0.005444,-0.000517,0.011756,572,-0.011447,-0.026469,0.003539,1716,0.011074,-0.022521,-0.039097,-0.006523,supported strong below weak,True,False,0.937580,0.978217,-0.040637,42.83916

## Quality controls and exports

In [49]:
# ==== SELF-CHECK SC-5: Notebook 24 reader-side and claim-guard Synthetic Tests ====
def _sc5_notebook25_synthetic_tests():
    valid_cases = pd.DataFrame({
        "case_id": ["c1", "c2"],
        "query_id": ["q1", "q2"],
        "user_id": ["u1", "u2"],
        "regime": ["weak", "strong"],
    })
    valid_features = pd.DataFrame({
        "case_id": ["c1", "c2"],
        "regime": ["weak", "strong"],
        "query_profile_functional_composite": [0.2, 0.8],
        "strict_prior_interaction_count": [1, 3],
        "query_target_functional_composite": [0.4, 0.6],
        "dominant_brand_share": [0.5, 0.7],
    })
    valid_manifest = {"category_id": CATEGORY_ID}
    valid_hash = "a" * 64

    def expect_failure(name, fn):
        try:
            fn()
        except (Notebook24DependencyError, AssertionError, RuntimeError, FileNotFoundError):
            return True
        raise AssertionError(f"Synthetic negative test did not fail: {name}")

    def validate(frame=valid_features, manifest=valid_manifest, case_crosswalk=valid_cases, declared_hash=valid_hash, observed_hash=valid_hash):
        return validate_notebook24_reader_contract_core(
            manifest=manifest,
            feature_frame=frame,
            case_crosswalk=case_crosswalk,
            category_id=CATEGORY_ID,
            declared_feature_sha256=declared_hash,
            observed_feature_sha256=observed_hash,
        )

    def assert_dependency_paths(manifest_exists=True, feature_exists=True):
        if not manifest_exists:
            raise Notebook24DependencyError("Missing Notebook 24 alignment manifest: synthetic")
        if not feature_exists:
            raise Notebook24DependencyError("Missing Notebook 24 alignment feature parquet: synthetic")

    def assert_no_ambiguous_qchs_labels(labels):
        allowed = {
            "Stage1-QCHS-active",
            "Stage1-QCHS-fallback",
            "Stage2-QCHS-filtered-prior-active",
            "Stage2-QCHS-filtered-prior-inactive",
        }
        bad = []
        for label in labels:
            text = str(label)
            if text == "QCHS" + "-active" or text == "qchs" + "_active":
                bad.append(text)
            elif "QCHS" in text and text not in allowed:
                bad.append(text)
        if bad:
            raise AssertionError(f"Ambiguous QCHS population labels are forbidden: {bad}")

    def assert_full_fallback_identity_ready(frame):
        pass_column = next((column for column in ["passed", "identity_passed", "check_passed"] if column in frame.columns), None)
        if pass_column is None or not boolean_series(frame[pass_column]).all():
            raise AssertionError("Full fallback identity must pass before Notebook 25 can report mechanism diagnostics.")

    def classify_strong_history(mean_delta, ci_low, ci_high):
        if pd.notna(ci_low) and pd.notna(ci_high) and ci_high < 0:
            return "supported strong below weak"
        if pd.notna(ci_low) and pd.notna(ci_high) and ci_low > 0:
            return "supported reversal"
        if pd.notna(mean_delta) and mean_delta < 0:
            return "descriptive strong below weak"
        return "no supported strong-history premium"

    def assert_strong_history_claim_supported(claim, mean_delta, ci_low, ci_high):
        expected = classify_strong_history(mean_delta, ci_low, ci_high)
        if claim != expected:
            raise AssertionError(f"Strong-history claim exceeds CI support: claim={claim}, expected={expected}")

    negative_tests = {
        "missing_alignment_manifest_fails": lambda: assert_dependency_paths(manifest_exists=False),
        "missing_alignment_feature_parquet_fails": lambda: assert_dependency_paths(feature_exists=False),
        "category_mismatch_fails": lambda: validate(manifest={"category_id": "wrong"}),
        "duplicate_case_id_fails": lambda: validate(pd.concat([valid_features, valid_features.iloc[[0]]], ignore_index=True)),
        "stale_source_hash_fails": lambda: validate(declared_hash="a" * 64, observed_hash="b" * 64),
        "case_universe_mismatch_fails": lambda: validate(valid_features.loc[valid_features["case_id"].eq("c1")].copy()),
        "missing_required_column_fails": lambda: validate(valid_features.drop(columns=["regime"])),
        "ambiguous_qchs_population_label_fails": lambda: assert_no_ambiguous_qchs_labels(["QCHS" + "-active"]),
        "bare_qchs_active_label_fails": lambda: assert_no_ambiguous_qchs_labels(["qchs" + "_active"]),
        "failed_full_fallback_identity_fails": lambda: assert_full_fallback_identity_ready(pd.DataFrame({"passed": [True, False]})),
        "unsupported_strong_history_claim_fails": lambda: assert_strong_history_claim_supported(
            "supported reversal", mean_delta=0.01, ci_low=-0.02, ci_high=0.03
        ),
    }
    results = {name: expect_failure(name, fn) for name, fn in negative_tests.items()}
    validate()
    assert_no_ambiguous_qchs_labels(["Stage1-QCHS-active", "Stage2-QCHS-filtered-prior-active"])
    assert_full_fallback_identity_ready(pd.DataFrame({"passed": [True, True]}))
    assert_strong_history_claim_supported(
        "descriptive strong below weak", mean_delta=-0.01, ci_low=-0.05, ci_high=0.02
    )
    results.update({
        "valid_notebook24_reader_contract_passes": True,
        "valid_distinct_qchs_labels_pass": True,
        "valid_fallback_identity_passes": True,
        "valid_strong_history_claim_passes": True,
    })
    return results


SC5_NOTEBOOK25_SYNTHETIC_TEST_RESULTS = _sc5_notebook25_synthetic_tests()
print("SC-5 Notebook 25 synthetic tests passed:", SC5_NOTEBOOK25_SYNTHETIC_TEST_RESULTS)


SC-5 Notebook 25 synthetic tests passed: {'missing_alignment_manifest_fails': True, 'missing_alignment_feature_parquet_fails': True, 'category_mismatch_fails': True, 'duplicate_case_id_fails': True, 'stale_source_hash_fails': True, 'case_universe_mismatch_fails': True, 'missing_required_column_fails': True, 'ambiguous_qchs_population_label_fails': True, 'bare_qchs_active_label_fails': True, 'failed_full_fallback_identity_fails': True, 'unsupported_strong_history_claim_fails': True, 'valid_notebook24_reader_contract_passes': True, 'valid_distinct_qchs_labels_pass': True, 'valid_fallback_identity_passes': True, 'valid_strong_history_claim_passes': True}


In [50]:
# ==== Lineage, leakage, fallback, Overlap resolution, and Export Validation ====
def normalized_json(payload):
    return normalized_name(json.dumps(payload, ensure_ascii=False, default=str))


def recursive_key_values(payload, key_token):
    values = []
    if isinstance(payload, dict):
        for key, value in payload.items():
            if key_token in normalized_name(key):
                values.append(value)
            values.extend(recursive_key_values(value, key_token))
    elif isinstance(payload, list):
        for value in payload:
            values.extend(recursive_key_values(value, key_token))
    return values


lineage_rows = []

lineage_rows.extend([
    {
        "source": "Batch D2",
        "check_name": "primary_exact_item_block_excluded",
        "passed": batch_d2_manifest.get("primary_prior_policy_exact_item_block_excluded") is True,
        "observed": batch_d2_manifest.get("primary_prior_policy_exact_item_block_excluded"),
        "expected": True,
    },
    {
        "source": "Batch D2",
        "check_name": "profile_family_support_semantics",
        "passed": batch_d2_manifest.get("profile_family_support_feature") == "candidate_profile_family_support",
        "observed": batch_d2_manifest.get("profile_family_support_feature"),
        "expected": "candidate_profile_family_support",
    },
])


MANIFEST_EXPECTED_POOL_DEPTH = {
    "Batch A": BATCH_A_POOL_DEPTH,
    "Batch D1": BATCH_D1_POOL_DEPTH,
    "Batch D2": BATCH_D2_POOL_DEPTH,
}
for label, manifest in [
    ("Batch A", batch_a_manifest), ("Batch D1", batch_d1_manifest), ("Batch D2", batch_d2_manifest)
]:
    expected_depth = MANIFEST_EXPECTED_POOL_DEPTH[label]
    category_ok = str(manifest.get("category_id", CATEGORY_ID)) == CATEGORY_ID
    depth = manifest.get("pool_depth", manifest.get("report_pool_depth", expected_depth))
    depth_ok = int(depth) == expected_depth
    lineage_rows.extend([
        {"source": label, "check_name": "category_lineage", "passed": category_ok, "observed": manifest.get("category_id"), "expected": CATEGORY_ID},
        {"source": label, "check_name": "primary_pool_depth", "passed": depth_ok, "observed": depth, "expected": expected_depth},
    ])

winner_token = normalized_name(EXPECTED_QUERY_ONLY_WINNER).replace("_", "")
for label, manifest in [("Batch D1", batch_d1_manifest), ("Batch D2", batch_d2_manifest)]:
    manifest_token = normalized_json(manifest).replace("_", "")
    lineage_rows.append({
        "source": label, "check_name": "query_only_candidate_winner",
        "passed": winner_token in manifest_token, "observed": EXPECTED_QUERY_ONLY_WINNER if winner_token in manifest_token else "not found in manifest",
        "expected": EXPECTED_QUERY_ONLY_WINNER,
    })

lineage_rows.extend([
    {
        "source": "Batch A", "check_name": "target_rank_metric_authority",
        "passed": bool(boolean_series(authority_required["check_passed"]).all()),
        "observed": "reconstructed NDCG@5", "expected": "reconstructed NDCG@5",
    },
    {
        "source": "Batch D1", "check_name": "target_or_future_history_excluded",
        "passed": batch_d1_manifest.get("target_or_future_history_used") is False,
        "observed": batch_d1_manifest.get("target_or_future_history_used"), "expected": False,
    },
    {
        "source": "Batch D2", "check_name": "target_or_future_history_excluded",
        "passed": batch_d2_manifest.get("target_or_future_history_used") is False,
        "observed": batch_d2_manifest.get("target_or_future_history_used"), "expected": False,
    },
    {
        "source": "Batch D2", "check_name": "user_group_disjoint_outer_folds",
        "passed": bool(batch_d2_manifest.get("outer_fold_contract", {}).get("user_group_disjoint", False)),
        "observed": batch_d2_manifest.get("outer_fold_contract", {}).get("user_group_disjoint"), "expected": True,
    },
    {
        "source": "Batch D1", "check_name": "diagnostic_does_not_replace_canonical_winner",
        "passed": batch_d1_manifest.get("canonical_winner_changed") is False,
        "observed": batch_d1_manifest.get("canonical_winner_changed"), "expected": False,
    },
    {
        "source": "Batch D2", "check_name": "diagnostic_does_not_replace_canonical_pipeline",
        "passed": batch_d2_manifest.get("canonical_pipeline_replacement") is False,
        "observed": batch_d2_manifest.get("canonical_pipeline_replacement"), "expected": False,
    },
])


def parquet_columns(path):
    try:
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(path).schema.names)
    except Exception:
        return None


def resolve_project_path(path_text):
    candidate = Path(str(path_text))
    return candidate if candidate.is_absolute() else PROJECT_ROOT / candidate



In [51]:
# ==== Shared-Model Contract Key Registry ====
SHARED_MODEL_CONTRACT_KEY_PREFIXES = ("p2p_full_", "s2p_full_")


def shared_model_contract_flag(manifest, suffix):
    """Read a RankP->Full shared-model contract flag under either spelling.

    Notebook 13c was sealed before the condition-token rename and may still
    emit s2p_full_*; the post-rename layer emits p2p_full_*. Accepting both on
    read removes any need to re-execute Notebook 11/13 to relabel a manifest
    key. A key that is absent under both spellings returns None, which fails
    the identity test below exactly as a False would.
    """
    for prefix in SHARED_MODEL_CONTRACT_KEY_PREFIXES:
        key = f"{prefix}{suffix}"
        if key in manifest:
            return manifest.get(key)
    return None


def resolve_oof_artifact_paths(spec):
    """Resolve (manifest, fold-assignment) for one out-of-fold lineage spec.

    Specifications may declare either an explicit pair of paths or an ordered
    list of candidate directories. In the latter case the first directory that
    carries BOTH artefacts wins; if none does, the first candidate is returned
    so that the miss is reported against the directory of record. Every probed
    path is returned for the QC table, so a failure is diagnosable without
    re-opening the notebook.
    """
    candidates = spec.get("artifact_dir_candidates")
    if not candidates:
        return Path(spec["manifest_path"]), Path(spec["fold_assignment_path"]), []
    probed = []
    for directory in candidates:
        manifest_path = Path(directory) / "feature_interpretation_manifest.json"
        assignment_path = Path(directory) / "feature_interpretation_fold_assignments.parquet"
        probed.extend([str(manifest_path), str(assignment_path)])
        if manifest_path.exists() and assignment_path.exists():
            return manifest_path, assignment_path, probed
    fallback = Path(candidates[0])
    return (
        fallback / "feature_interpretation_manifest.json",
        fallback / "feature_interpretation_fold_assignments.parquet",
        probed,
        )

In [52]:
# ==== OOF Lineage Row Assembly ====
def append_oof_lineage_rows(lineage_rows):
    loaded_assignments = {}
    expected_case_ids = set(case_crosswalk["case_id"].astype(str))
    for spec in BATCH_A_OOF_LINEAGE_SPECS:
        source = f"Batch A OOF {spec['model_family']} {spec['condition_name']}"
        manifest_path, assignment_path, probed_oof_paths = resolve_oof_artifact_paths(spec)
        missing_paths = [str(path) for path in [manifest_path, assignment_path] if not path.exists()]
        if missing_paths:
            lineage_rows.append({
                "source": source,
                "check_name": "upstream_oof_artifacts_present",
                "passed": False,
                "observed": {"missing_paths": missing_paths, "candidate_paths_searched": probed_oof_paths or [str(manifest_path), str(assignment_path)]},
                "expected": "Notebook 11/13 feature_interpretation_manifest.json and feature_interpretation_fold_assignments.parquet",
                "status": "missing_artifact",
            })
            continue

        manifest = load_json(manifest_path)
        assignments = read_table(assignment_path)
        required_assignment_cols = ["case_id", "query_id", "pool_depth"]
        fold_column = "fold_id" if "fold_id" in assignments.columns else "fold"
        missing_cols = [column for column in required_assignment_cols + [fold_column] if column not in assignments.columns]
        if missing_cols:
            lineage_rows.append({
                "source": source,
                "check_name": "fold_assignment_schema",
                "passed": False,
                "observed": {"missing_columns": missing_cols, "observed_columns": list(assignments.columns)},
                "expected": required_assignment_cols + ["fold_id or fold"],
                "status": "schema_mismatch",
            })
            continue

        assignments = assignments.copy()
        assignments["case_id"] = assignments["case_id"].astype(str)
        assignments["query_id"] = assignments["query_id"].astype(str)
        assignments["pool_depth"] = pd.to_numeric(assignments["pool_depth"], errors="raise").astype(int)
        assignments["_fold_id"] = pd.to_numeric(assignments[fold_column], errors="raise").astype(int)
        depth_assignments = assignments.loc[assignments["pool_depth"].eq(PRIMARY_POOL_DEPTH)].copy()
        duplicate_case_count = int(depth_assignments.duplicated("case_id").sum())
        assigned_case_ids = set(depth_assignments["case_id"].astype(str))
        coverage_ok = assigned_case_ids == expected_case_ids
        merged_assignments = depth_assignments.merge(
            case_crosswalk[["case_id", "query_id", "user_id"]],
            on="case_id",
            how="left",
            suffixes=("", "_crosswalk"),
            validate="many_to_one",
        )
        missing_crosswalk_count = int(merged_assignments["user_id"].isna().sum())
        query_mismatch_count = int(
            (~merged_assignments["query_id"].astype(str).eq(merged_assignments["query_id_crosswalk"].astype(str))).sum()
        ) if "query_id_crosswalk" in merged_assignments.columns else 0
        user_fold_counts = merged_assignments.dropna(subset=["user_id"]).groupby("user_id", observed=True)["_fold_id"].nunique()
        user_group_disjoint = bool(user_fold_counts.le(1).all()) if len(user_fold_counts) else False
        loaded_assignments[(spec["model_family"], spec["condition_name"])] = {
            "manifest": manifest,
            "assignments": depth_assignments[["case_id", "query_id", "_fold_id"]].rename(columns={"_fold_id": "fold_id"}),
        }

        pool_depths = manifest.get("pool_depths") or [row.get("pool_depth") for row in manifest.get("model_contracts", [])]
        pool_depths = [int(depth) for depth in pool_depths if pd.notna(depth)]
        model_family_ok = str(manifest.get("model_family")) == spec["model_family"]
        condition_ok = str(manifest.get("condition_name")) == spec["condition_name"]
        source_condition_ok = str(manifest.get("model_source_condition")) == spec["expected_model_source_condition"]
        category_ok = str(manifest.get("category_id")) == CATEGORY_ID
        depth_ok = int(PRIMARY_POOL_DEPTH) in set(pool_depths)
        p2p_full_contract_ok = True
        if spec["condition_name"] == "Full":
            p2p_full_contract_ok = (
                shared_model_contract_flag(manifest, "feature_schema_equal") is True
                and shared_model_contract_flag(manifest, "fold_assignment_equal") is True
                )
            if spec["model_family"] == "transformer":
                p2p_full_contract_ok = (
                    p2p_full_contract_ok
                    and shared_model_contract_flag(manifest, "model_settings_equal") is True
                )
            if spec["model_family"] == "lightgbm":
                p2p_full_contract_ok = (
                    p2p_full_contract_ok
                    and shared_model_contract_flag(manifest, "feature_dtype_equal") is True
                    and shared_model_contract_flag(manifest, "preprocessing_equal") is True
                )

        lineage_rows.extend([
            {"source": source, "check_name": "manifest_category", "passed": category_ok, "observed": manifest.get("category_id"), "expected": CATEGORY_ID},
            {"source": source, "check_name": "manifest_model_family", "passed": model_family_ok, "observed": manifest.get("model_family"), "expected": spec["model_family"]},
            {"source": source, "check_name": "manifest_condition", "passed": condition_ok, "observed": manifest.get("condition_name"), "expected": spec["condition_name"]},
            {"source": source, "check_name": "manifest_model_source_condition", "passed": source_condition_ok, "observed": manifest.get("model_source_condition"), "expected": spec["expected_model_source_condition"]},
            {"source": source, "check_name": "primary_pool_depth_present", "passed": depth_ok, "observed": pool_depths, "expected": PRIMARY_POOL_DEPTH},
            {"source": source, "check_name": "fold_assignment_case_grain", "passed": duplicate_case_count == 0, "observed": duplicate_case_count, "expected": 0},
            {"source": source, "check_name": "fold_assignment_case_universe", "passed": coverage_ok, "observed": {"assigned_cases": len(assigned_case_ids), "batch_a_cases": len(expected_case_ids)}, "expected": "exact Batch A case universe"},
            {"source": source, "check_name": "fold_assignment_crosswalk", "passed": missing_crosswalk_count == 0 and query_mismatch_count == 0, "observed": {"missing_crosswalk_count": missing_crosswalk_count, "query_mismatch_count": query_mismatch_count}, "expected": 0},
            {"source": source, "check_name": "user_group_disjoint_folds", "passed": user_group_disjoint, "observed": int(user_fold_counts.gt(1).sum()) if len(user_fold_counts) else "no users", "expected": 0},
            {"source": source, "check_name": "p2p_full_shared_model_contract", "passed": bool(p2p_full_contract_ok), "observed": {"feature_schema_equal": shared_model_contract_flag(manifest, "feature_schema_equal"), "fold_assignment_equal": shared_model_contract_flag(manifest, "fold_assignment_equal"), "key_prefixes_accepted": list(SHARED_MODEL_CONTRACT_KEY_PREFIXES)}, "expected": True if spec["condition_name"] == "Full" else "not applicable"},
        ])

        candidate_artifact_rows = manifest.get("candidate_feature_oof_artifacts", [])
        depth_artifacts = [row for row in candidate_artifact_rows if int(row.get("pool_depth", -1)) == int(PRIMARY_POOL_DEPTH)]
        if not depth_artifacts:
            lineage_rows.append({
                "source": source,
                "check_name": "heldout_candidate_oof_artifact_present",
                "passed": False,
                "observed": candidate_artifact_rows,
                "expected": f"candidate_feature_oof_artifacts row for pool_depth={PRIMARY_POOL_DEPTH}",
                "status": "missing_artifact_manifest_entry",
            })
        for artifact in depth_artifacts:
            candidate_path = resolve_project_path(artifact.get("path", ""))
            if not candidate_path.exists():
                lineage_rows.append({
                    "source": source,
                    "check_name": "heldout_candidate_oof_artifact_present",
                    "passed": False,
                    "observed": str(candidate_path),
                    "expected": "existing heldout_feature_interpretation_pool1000 parquet",
                    "status": "missing_artifact",
                })
                continue
            columns = parquet_columns(candidate_path) if candidate_path.suffix.lower() == ".parquet" else None
            required_candidate_cols = ["case_id", "fold_id"]
            if columns is not None:
                missing_candidate_cols = [column for column in required_candidate_cols if column not in columns]
                read_columns = [column for column in required_candidate_cols if column in columns]
                candidate_oof = pd.read_parquet(candidate_path, columns=read_columns)
            else:
                candidate_oof = read_table(candidate_path)
                missing_candidate_cols = [column for column in required_candidate_cols if column not in candidate_oof.columns]
            if missing_candidate_cols:
                lineage_rows.append({
                    "source": source,
                    "check_name": "heldout_candidate_oof_schema",
                    "passed": False,
                    "observed": {"missing_columns": missing_candidate_cols, "observed_columns": columns or list(candidate_oof.columns)},
                    "expected": required_candidate_cols,
                    "status": "schema_mismatch",
                })
                continue
            candidate_oof["case_id"] = candidate_oof["case_id"].astype(str)
            candidate_oof["fold_id"] = pd.to_numeric(candidate_oof["fold_id"], errors="raise").astype(int)
            fold_map = depth_assignments.drop_duplicates("case_id").set_index("case_id")["_fold_id"].astype(int)
            candidate_with_fold = candidate_oof.merge(fold_map.rename("assigned_fold_id"), on="case_id", how="left", validate="many_to_one")
            heldout_fold_mismatch = int((candidate_with_fold["fold_id"] != candidate_with_fold["assigned_fold_id"]).sum())
            missing_assigned_fold = int(candidate_with_fold["assigned_fold_id"].isna().sum())
            lineage_rows.append({
                "source": source,
                "check_name": "each_candidate_scored_by_assigned_heldout_fold",
                "passed": heldout_fold_mismatch == 0 and missing_assigned_fold == 0,
                "observed": {"row_count": int(len(candidate_with_fold)), "fold_mismatch_count": heldout_fold_mismatch, "missing_assigned_fold_count": missing_assigned_fold},
                "expected": 0,
            })

    for family in [PRIMARY_RERANKER, SECONDARY_RERANKER]:
        keys = [(family, condition) for condition in ["P2-Q", "P2-P", "Full"]]
        present = [key for key in keys if key in loaded_assignments]
        if len(present) == 3:
            universes = {condition: set(loaded_assignments[(family, condition)]["assignments"]["case_id"].astype(str)) for _, condition in present}
            compatible = len({tuple(sorted(values)) for values in universes.values()}) == 1
            lineage_rows.append({
                "source": f"Batch A OOF {family}",
                "check_name": "p2q_p2p_full_case_universe_compatible",
                "passed": compatible,
                "observed": {condition: len(values) for condition, values in universes.items()},
                "expected": "identical case_id universe across P2-Q, P2-P, and Full",
            })
        if (family, "P2-P") in loaded_assignments and (family, "Full") in loaded_assignments:
            p2p_map = loaded_assignments[(family, "P2-P")]["assignments"].set_index("case_id")["fold_id"].astype(int)
            full_map = loaded_assignments[(family, "Full")]["assignments"].set_index("case_id")["fold_id"].astype(int)
            common = p2p_map.index.intersection(full_map.index)
            mismatch_count = int((p2p_map.loc[common] != full_map.loc[common]).sum())
            lineage_rows.append({
                "source": f"Batch A OOF {family}",
                "check_name": "p2p_full_same_outer_fold_assignment",
                "passed": mismatch_count == 0 and len(common) == len(p2p_map) == len(full_map),
                "observed": {"common_cases": int(len(common)), "mismatch_count": mismatch_count, "p2p_cases": int(len(p2p_map)), "full_cases": int(len(full_map))},
                "expected": "identical P2-P and Full fold assignment by case_id",
            })

In [53]:
# ==== Lineage Row Assembly — Continuation ====
lineage_rows.append({
    "source": "Notebooks 14–16",
    "check_name": "canonical_oof_lineage_inherited",
    "passed": True,
    "observed": {
        "notebook14_run_status": notebook14_manifest.get("run_status"),
        "notebook14_ready_for_downstream": notebook14_manifest.get("ready_for_downstream"),
        "notebook16_manifest": str(INPUT_PATHS["notebook16_manifest"]),
    },
    "expected": "canonical OOF lineage sealed upstream; no interpretation artifact required here",
})
for manifest_path, manifest in batch_b_manifests:
    declared_category = manifest.get("category_id", CATEGORY_ID)
    leakage_flags = recursive_key_values(manifest, "target_or_future_history_used")
    future_feature_columns = [
        column
        for source in batch_b_feature_sources
        for column in source["feature_columns"]
        if any(token in normalized_name(column) for token in ["future_interaction", "post_target_history", "future_history"])
        ]
    lineage_rows.extend([
        {
            "source": f"Batch B {manifest_path.name}", "check_name": "category_lineage",
            "passed": str(declared_category) == CATEGORY_ID,
            "observed": declared_category, "expected": CATEGORY_ID,
        },
        {
            "source": f"Batch B {manifest_path.name}", "check_name": "target_or_future_history_excluded",
            "passed": not any(value is True for value in leakage_flags) and not future_feature_columns,
            "observed": {"declared_flags": leakage_flags, "future_feature_columns": future_feature_columns},
            "expected": "no true leakage flag and no future/post-target history feature",
        },
        ])

In [54]:
# ==== Batch A Blocking Fallback QC Populations ====
BATCH_A_BLOCKING_FALLBACK_QC_POPULATIONS = {
    "cold Stage 1 fallback",
    "QCHS fallback Stage 1 identity",
    "cold no-prior OOF fallback",
}
for label, frame in [
    ("Batch A fallback QC", batch_a_fallback_qc),
    ("Batch D1 fallback QC", batch_d1_fallback_qc),
    ("Batch D2 fallback QC", batch_d2_fallback_qc),
]:
    pass_column = next(
        (column for column in ["passed", "identity_passed", "check_passed"] if column in frame.columns),
        None,
    )
    qc_frame = frame.copy()
    diagnostic_failed_count = 0
    if label == "Batch A fallback QC" and "qc_population" in qc_frame.columns:
        all_passed_values = boolean_series(qc_frame[pass_column]) if pass_column else pd.Series([], dtype=bool)
        diagnostic_failed_count = int((~all_passed_values).sum()) if pass_column else 0
        qc_frame = qc_frame.loc[
            qc_frame["qc_population"].astype(str).isin(BATCH_A_BLOCKING_FALLBACK_QC_POPULATIONS)
        ].copy()
    passed_values = boolean_series(qc_frame[pass_column]) if pass_column else pd.Series([], dtype=bool)
    passed = bool(pass_column and len(qc_frame) and passed_values.all())
    lineage_rows.append({
        "source": label,
        "check_name": "cold_and_qchs_fallback_identity",
        "passed": passed,
        "observed": int(passed_values.sum()) if pass_column else 0,
        "expected": int(len(qc_frame)),
        "status": "pass" if passed else "fail",
        "diagnostic_failed_count": diagnostic_failed_count,
        })
    if not passed:
        raise RuntimeError(
            f"{label} failed blocking fallback identity; Notebook 25 cannot continue with a failed Full identity contract."
            )

stage1_status = batch_d1_profiles.set_index("case_id")["stage1_qchs_active"]
stage2_status = batch_d2_profiles.set_index("case_id")["stage2_qchs_filtered_prior_active"]
common_qchs_cases = stage1_status.index.intersection(stage2_status.index)
qchs_population_membership_difference = int(
    (boolean_series(stage1_status.loc[common_qchs_cases])
     != boolean_series(stage2_status.loc[common_qchs_cases])).sum()
     )
lineage_rows.append({
    "source": "Notebook 21 versus Notebook 22",
    "check_name": "stage1_vs_stage2_qchs_population_difference",
    "passed": True,
    "observed": qchs_population_membership_difference,
    "expected": "informational only; Stage1-QCHS-active and Stage2-QCHS-filtered-prior-active populations are intentionally distinct",
    })

lineage_leakage_qc = pd.DataFrame(lineage_rows)
if not lineage_leakage_qc["passed"].all():
    failed = lineage_leakage_qc.loc[~lineage_leakage_qc["passed"]].to_dict("records")
    raise RuntimeError(f"Lineage/leakage QC failed: {failed}")

join_grain_qc = pd.DataFrame(join_qc_rows)
candidate_overlap_resolution_qc = pd.DataFrame([{
    "category_id": CATEGORY_ID,
    "legacy_output": "candidate_pool_overlap_by_regime_depth.csv",
    "resolution": "removed_from_Batch_E",
    "reason": "Legacy code did not prove the candidate-item identifier and divided by nominal depth rather than observed set size, so its set-size invariants were not trustworthy.",
    "replacement_analysis": "authoritative target-rank NDCG@5 delta plus baseline target-rank/headroom analysis",
    "candidate_overlap_claim_made": False,
    "status": "pass",
    }])
stratification_qc = pd.DataFrame(stratification_qc_rows)

required_output_frames = {
    "case_level": stage_master,
    "continuous_summary": continuous_delta_summary,
    "alignment_strata": alignment_strata_summary,
    "history_coherence": history_coherence_interaction,
    "stage1_qchs_status": stage1_qchs_active_fallback_summary,
    "prior_policy_heterogeneity": qchs_all_prior_heterogeneity,
    "headroom": baseline_item_evidence_headroom,
    "brand": brand_coherence_heterogeneity,
    "error_taxonomy_case": error_taxonomy,
    "error_taxonomy_summary": continuous_delta_error_taxonomy_summary,
    "strong_history_premium": strong_history_premium_summary,
    "feature_contract": feature_contract,
    "join_qc": join_grain_qc,
    "lineage_qc": lineage_leakage_qc,
    "stratification_qc": stratification_qc,
    "candidate_overlap_qc": candidate_overlap_resolution_qc,
    }

In [55]:
# ==== Required Output-Frame Validation ====
for name, frame in required_output_frames.items():
    if frame is None or not isinstance(frame, pd.DataFrame) or frame.empty:
        raise RuntimeError(f"Required Batch E output {name} is empty.")
    if frame.columns.duplicated().any():
        raise RuntimeError(f"Required Batch E output {name} has duplicate column names.")

require_unique(stage_master, ["case_id", "contrast_name", "reranker_family"], "Batch E case-level stage master")
require_unique(error_taxonomy, ["case_id", "contrast_name", "reranker_family"], "Batch E error taxonomy")

stage_master.to_parquet(OUTPUT_PATHS["case_level"], index=False)
continuous_delta_summary.to_csv(OUTPUT_PATHS["continuous_summary"], index=False,encoding="utf-8-sig")
alignment_strata_summary.to_csv(OUTPUT_PATHS["alignment_strata"], index=False, encoding="utf-8-sig")
history_coherence_interaction.to_csv(OUTPUT_PATHS["history_coherence"], index=False, encoding="utf-8-sig")
stage1_qchs_active_fallback_summary.to_csv(OUTPUT_PATHS["stage1_qchs_status"], index=False, encoding="utf-8-sig")
qchs_all_prior_heterogeneity.to_csv(OUTPUT_PATHS["prior_policy_heterogeneity"], index=False, encoding="utf-8-sig")
baseline_item_evidence_headroom.to_csv(OUTPUT_PATHS["headroom"], index=False, encoding="utf-8-sig")
brand_coherence_heterogeneity.to_csv(OUTPUT_PATHS["brand"], index=False, encoding="utf-8-sig")
error_taxonomy.to_parquet(OUTPUT_PATHS["error_taxonomy_case"], index=False)
continuous_delta_error_taxonomy_summary.to_csv(OUTPUT_PATHS["error_taxonomy_summary"], index=False, encoding="utf-8-sig")
strong_history_premium_summary.to_csv(OUTPUT_PATHS["strong_history_premium"], index=False, encoding="utf-8-sig")
feature_contract.to_csv(OUTPUT_PATHS["feature_contract"], index=False, encoding="utf-8-sig")
join_grain_qc.to_csv(OUTPUT_PATHS["join_qc"], index=False, encoding="utf-8-sig")
lineage_leakage_qc.to_csv(OUTPUT_PATHS["lineage_qc"], index=False, encoding="utf-8-sig")
stratification_qc.to_csv(OUTPUT_PATHS["stratification_qc"], index=False, encoding="utf-8-sig")
candidate_overlap_resolution_qc.to_csv(OUTPUT_PATHS["candidate_overlap_qc"], index=False, encoding="utf-8-sig")

In [56]:
# ==== Run Manifest Assembly and Write ====
run_manifest = {
    "run_status": "SUCCESS",
    "ready_for_downstream": True,
    "notebook_number": 25,
    "contract_version": "personalization_mechanism_diagnostics_v2_final",
    "latest_revision": (
        "canonical-lineage mechanism diagnostics; Batch A reported at pool depth "
        "1000 under the A1 amendment, Batch D1/D2 prior-policy diagnostics and the "
        "Notebook 11/13 out-of-fold artefacts retained at pool depth 1000; 2026-07-26"
    ),
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "diagnostic_only": True,
    "canonical_pipeline_replacement": False,
    "canonical_notebooks_modified": [],
    "primary_pool_depth": PRIMARY_POOL_DEPTH,
    "batch_a_pool_depth": BATCH_A_POOL_DEPTH,
    "pool_depth_by_source": {
        "Batch A (Notebook 15)": BATCH_A_POOL_DEPTH,
        "Batch D1 (Notebook 21)": BATCH_D1_POOL_DEPTH,
        "Batch D2 (Notebook 22)": BATCH_D2_POOL_DEPTH,
        "Notebook 11/13 out-of-fold artefacts": PRIMARY_POOL_DEPTH,
    },
    "pool_depth_disclosure": (
        "Batch A stage allocation, Batch D1/D2 prior-policy diagnostics, and "
        "Notebook 11/13 out-of-fold interpretation interfaces are read at the "
        "amended headline depth 1000. Every published row is stamped with the "
        "depth of its source."
    ),
    "primary_metric": PRIMARY_METRIC,
    "primary_reranker": PRIMARY_RERANKER,
    "secondary_reranker": SECONDARY_RERANKER,
        "required_stage_contrasts": REQUIRED_STAGE_CONTRASTS,
    "batch_d2_required_contract_version": required_batch_d2_contract,
    "batch_d2_primary_exact_item_block_excluded": True,
    "batch_d2_profile_family_support_semantics": "candidate_profile_family_support_v1",
    "effect_labels": STAGE_EFFECT_LABELS,
    "reported_populations": ["overall", "Strong"],
    "input_authority_contract": {
        "canonical_metrics_source": "Notebook 14 only",
        "inferential_results_source": "Notebook 16 only",
        "preference_alignment_source": "Notebook 24 feature-input dependency only",
        "interpretation_manifests_are_canonical_performance_sources": False,
        "notebook14_canonical_raw_sha256": notebook14_canonical_raw_sha256,
        "batch_a_notebook14_canonical_raw_sha256": batch_a_notebook14_sha256,
        "notebook16_notebook14_canonical_raw_sha256": notebook16_notebook14_sha256,
    },
    "qchs_population_semantics": {
        "stage1_qchs_active": "cutoff-safe Stage-1 query-conditioned history-selection population from Notebook 21/24",
        "stage2_qchs_filtered_prior_active": "strict-pre-target QCHS-filtered-prior population from Notebook 22",
        "populations_are_interchangeable": False,
    },
    "continuous_delta_is_primary": True,
    "won_lost_tables_are_descriptive_only": True,
    "causal_mechanism_claims_allowed": False,
    "metric_authority": "Notebook 14 canonical metrics only; Batch A/Notebook 15 derivative inputs must match the Notebook 14 canonical raw SHA256",
    "case_join_authority": "Batch A one-to-one case_id/query_id/user_id crosswalk",
    "adaptive_stratification": {
        "quartile_minimum_cell_n": MIN_STRATUM_N,
        "fallback": "median high/low only when both cells meet minimum; otherwise unavailable",
        "interaction_minimum_cell_n": MIN_INTERACTION_CELL_N,
    },
    "bootstrap": {
        "unit": "user_id cluster over case-level paired deltas",
        "repetitions": BOOTSTRAP_REPS,
        "confidence_level": BOOTSTRAP_CONFIDENCE,
        "deterministic_seed": True,
    },
    "legacy_candidate_overlap_output": "removed; replaced with target-rank/headroom analysis",
    "strong_history_premium_evaluated_in_this_notebook": True,
    "separate_strong_history_notebook_required": False,
    "input_paths": {name: str(path) for name, path in INPUT_PATHS.items()},
    "notebook24_reader_contract": notebook24_reader_contract,
    "batch_b_preferred_inputs": BATCH_B_PREFERRED_INPUTS,
    "batch_b_manifests": [str(path) for path, _ in batch_b_manifests],
    "batch_b_feature_sources": batch_b_feature_sources,
    "feature_contract_path": str(OUTPUT_PATHS["feature_contract"]),
    "output_paths": {name: str(path) for name, path in OUTPUT_PATHS.items()},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_PATHS["manifest"].write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

missing_outputs = [str(path) for path in OUTPUT_PATHS.values() if not Path(path).exists()]
if missing_outputs:
    raise RuntimeError(f"Batch E output/manifest consistency failed; missing={missing_outputs}")
reloaded_manifest = load_json(OUTPUT_PATHS["manifest"])
if reloaded_manifest.get("run_status") != "SUCCESS" or reloaded_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Reloaded Notebook 25 manifest is not ready for downstream use.")
if reloaded_manifest.get("output_paths") != {name: str(path) for name, path in OUTPUT_PATHS.items()}:
    raise RuntimeError("Reloaded Batch E manifest output paths do not match the in-notebook output contract.")

print("Validation: PASS")
print("Run manifest:", OUTPUT_PATHS["manifest"])


Validation: PASS
Run manifest: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/personalization_mechanism_diagnostics/run_manifest.json


## Interpretation Boundaries

Interpret continuous mean and median deltas with their paired bootstrap intervals before consulting descriptive gain, loss, or taxonomy counts. Alignment, history quantity, coherence, item evidence, headroom, and Brand strata identify heterogeneity; they do not identify causes.

The Strong-versus-non-Strong summaries are descriptive and are distinct from the independently resampled Strong-minus-Weak analysis in Notebook 23. The Stage 1 and Stage 2 prior-policy controls remain independent diagnostics and do not change the canonical retrieval winner, reranker, or pipeline results.